# CALCE V–SoC — Source-aligned five-stage scientific rebuild

**Source:** user-uploaded `ConditionvCodes(2).py` (32,549 logical lines). Scientific functions and estimators have been copied from the indicated exact line ranges. Only automatically executed Colab upload-entrypoints and other exported execution entrypoints were omitted; each block is invoked via local, existing Colab inputs. No R2/R2.2/R4 reconstruction reruns.

**Dependency order:** 01 band coverage → 02 age sensitivity; 03 unified → 04 unified coverage QA and 05 five representations. The two branches may be run independently; results are evaluated against the frozen manuscript before final publication packaging.

**Inputs must already exist at the verified paths.** ZIP archives are written to `/content`, so the final publisher can find them. Previously saved archives are never silently replaced. Each scientific stage remains unmodified in its computational routines.


## 00 — Inputs and safe output locations

In [ ]:
from pathlib import Path
import hashlib, os, shutil, json

SOURCE_DIR = Path('/content/revision_R2_2_global')
BROAD_DIR = Path('/content/outputs/R4B_broad_52_132')
REBUILD_DIR = Path('/content/SoC_Rebuild_ConditionvCodes2')
REBUILD_DIR.mkdir(exist_ok=True)

INPUT_CYCLES = SOURCE_DIR / 'global_cycle_profiles.pkl'
INPUT_CELLS = SOURCE_DIR / 'cell_profiles_nominal_concordant.csv'
INPUT_BROAD = BROAD_DIR / 'broad_52_132_cell_profiles.csv'


def require_path(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Required input missing: {path}')
    return path


def stage_dir(stage):
    path = REBUILD_DIR / stage
    if path.exists() and any(path.iterdir()):
        raise FileExistsError(f'Nonempty stage directory (refuse to overwrite): {path}')
    path.mkdir(parents=True, exist_ok=True)
    return path


def stage_zip(stage, prefix):
    folder = REBUILD_DIR / stage
    path = Path('/content') / f'{prefix}ConditionvCodes2.zip'
    if path.exists():
        raise FileExistsError(f'Refuse to replace existing scientific archive: {path}')
    archive = Path(shutil.make_archive(str(path.with_suffix('')), 'zip', root_dir=str(folder)))
    with __import__('zipfile').ZipFile(archive) as zf:
        if zf.testzip() is not None:
            raise RuntimeError(f'Archive CRC failed: {archive}')
    print('STAGE FILES:', folder)
    print('ARCHIVE:', archive)
    print('SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
    return archive

for name, file in [('Nominal cycles',INPUT_CYCLES), ('Full cell profiles',INPUT_CELLS), ('Window cell profiles',INPUT_BROAD)]:
    print(name, 'FOUND' if file.is_file() else 'MISSING', str(file))


## 01 — 80% SoC-band coverage and reference inference

Original `ConditionvCodes(2).py` lines **30,073–30,393**. Run after the prior required stage.

In [ ]:
# GOOGLE COLAB: Tamamini TEK hucrede calistirin.
# Girdi: kendi R2.2 cikti dosyaniz global_cycle_profiles.pkl.
# Istege bagli: eski R7 kontrast CSV'si (delta_age_adjusted_V sutunlu).
import io, json, hashlib, tempfile, shutil, warnings, platform
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from threadpoolctl import threadpool_limits

N_BOOT=2000
SEED=20260927
REFERENCE_CYCLE=108
RULES={'legacy_1node':None,'coverage_60':0.60,'coverage_80':0.80,'coverage_100':1.0}
BANDS=[('0-20',0,20),('20-60',20,60),('60-90',60,90)]
BOOT_RULES={'legacy_1node','coverage_80'}
MIN_BOOT_VALID_FRACTION=0.95
# Ayni R7 modeli ve pratik siniflar; yeni yas modeli bu blokta YOK.
META=pd.DataFrame([(1+6*t+2*r+k,T,c,rate)
    for t,T in enumerate((10,25,45,60))
    for r,rate in enumerate(('0.7C','1C','2C'))
    for k,c in enumerate(('C/5','C/40'))],
    columns=['Condition_ID','Temperature','Rate','Charge_Rate'])
LOOKUP={(r.Temperature,r.Rate,r.Charge_Rate):r.Condition_ID for r in META.itertuples()}


def specifications():
    rows=[]
    def add(f,i,l,r):
        rows.append(dict(contrast_family=f,contrast_id=i,lhs_condition=LOOKUP[l],rhs_condition=LOOKUP[r]))
    for T in (10,25,45,60):
        add('cutoff_C40_minus_C5',f'cutoff_T{T}_DR1C',(T,'C/40','1C'),(T,'C/5','1C'))
    for T in (10,25,45,60):
        add('rate_0.7C_minus_1C',f'rate07minus1_T{T}_C40',(T,'C/40','0.7C'),(T,'C/40','1C'))
    for dr in ('0.7C','1C','2C'):
        add('temperature_60_minus_25',f'temp60minus25_C40_DR{dr}',(60,'C/40',dr),(25,'C/40',dr))
    for family,prefix,lr,rr in [('rate_2C_minus_1C','rate2minus1','2C','1C'),
                                ('rate_0.7C_minus_2C','rate07minus2','0.7C','2C')]:
        for T in (10,25,45,60):
            for c in ('C/5','C/40'):
                add(family,f'{prefix}_T{T}_{c.replace("/","")}',(T,c,lr),(T,c,rr))
    assert len(rows)==27
    return rows
SPECS=specifications()


def sign(x):
    return 'NA' if not np.isfinite(x) else 'positive' if x>1e-12 else 'negative' if x< -1e-12 else 'zero'


def bucket(x):
    if not np.isfinite(x): return 'NA'
    return ('negligible' if abs(x)<.05 else 'small' if abs(x)<.10 else 'moderate' if abs(x)<.15 else 'large')


def load_cycles(blob):
    # Pickle kendi uretilmis R2.2 dosyaniz olmalidir.
    d=pd.read_pickle(io.BytesIO(blob))
    required=['sample_int','Condition_ID','Cycle_Index','matches_nominal']+[f'V_{i:03d}' for i in range(90)]
    if not isinstance(d,pd.DataFrame) or not set(required).issubset(d):
        raise ValueError('R2.2 global_cycle_profiles.pkl semasi bekleniyor.')
    before=len(d)
    if d.matches_nominal.isna().any() or not d.matches_nominal.isin([True,False]).all():
        raise ValueError('matches_nominal boolean olmali.')
    d=d[d.matches_nominal.eq(True)].copy()
    if d.empty: raise ValueError('Nominal uyumlu cevrim yok.')
    for c in required[:3]:
        d[c]=pd.to_numeric(d[c],errors='raise')
        a=d[c].to_numpy(float)
        if not np.isfinite(a).all() or not np.allclose(a,np.round(a),rtol=0,atol=1e-8):
            raise ValueError(f'{c}: sonlu tamsayi bekleniyor.')
        d[c]=np.round(a).astype(int)
    if not d.Condition_ID.between(1,24).all() or (d.Cycle_Index<0).any():
        raise ValueError('Kosul veya cevrim araligi hatali.')
    if d.duplicated(required[:3]).any(): raise ValueError('Yinelenen hucre-kosul-cevrim var; otomatik silinmedi.')
    if (d.groupby('sample_int').Condition_ID.nunique()>1).any(): raise ValueError('Hucre birden cok kosulda.')
    for r in d[['sample_int','Condition_ID']].drop_duplicates().itertuples():
        allowed=range(8*(r.Condition_ID-1)+1,8*r.Condition_ID+1) if r.Condition_ID<=22 else (range(177,180) if r.Condition_ID==23 else range(185,188))
        if r.sample_int not in allowed: raise ValueError(f'Nominal hucre haritasi uyusmazligi: {r}')
    for c in required[4:]:
        d[c]=pd.to_numeric(d[c],errors='raise')
        if np.isinf(d[c]).any(): raise ValueError(f'{c}: sonsuz gerilim var.')
    print(f'Girdi: {before:,}; nominal uyumlu: {len(d):,} cevrim; {d.sample_int.nunique()} hucre.')
    return d.reset_index(drop=True),before


def band_data(d,band,lo,hi):
    A=d[[f'V_{i:03d}' for i in range(lo,hi)]].to_numpy(float)
    finite=np.isfinite(A);n=finite.sum(axis=1)
    q=d[['sample_int','Condition_ID','Cycle_Index']].copy()
    q['SoC_band']=band;q['n_band_nodes']=hi-lo;q['n_valid']=n
    q['coverage_fraction']=n/(hi-lo)
    q['Voltage_band']=np.divide(np.nansum(A,axis=1),n,out=np.full(len(A),np.nan),where=n>0)
    q['age_c']=(q.Cycle_Index-REFERENCE_CYCLE)/100
    for rule,frac in RULES.items():
        q[rule]=n >= (1 if frac is None else int(np.ceil(frac*(hi-lo))))
    return q


def coverage_tables(q):
    rows=[];cellrows=[]
    for rule in RULES:
        for cid in range(1,25):
            a=q[q.Condition_ID.eq(cid)];b=a[a[rule]];old=a[a.legacy_1node]
            rows.append(dict(rule=rule,SoC_band=q.SoC_band.iloc[0],Condition_ID=cid,
                n_cycles_input=len(a),n_cycles_legacy=len(old),n_cycles_kept=len(b),
                n_cycles_removed_vs_legacy=len(old)-len(b),n_cycles_no_band_nodes=int(a.n_valid.eq(0).sum()),
                n_cells_input=a.sample_int.nunique(),n_cells_legacy=old.sample_int.nunique(),n_cells_kept=b.sample_int.nunique(),
                n_cells_lost_vs_legacy=old.sample_int.nunique()-b.sample_int.nunique(),
                cycle_min=b.Cycle_Index.min(),cycle_median=b.Cycle_Index.median(),cycle_max=b.Cycle_Index.max(),
                coverage_min=b.coverage_fraction.min(),coverage_median=b.coverage_fraction.median()))
        for sid,a in q.groupby('sample_int'):
            b=a[a[rule]]
            cellrows.append(dict(rule=rule,SoC_band=q.SoC_band.iloc[0],sample_int=sid,
                Condition_ID=int(a.Condition_ID.iloc[0]),n_cycles_input=len(a),
                n_cycles_legacy=int(a.legacy_1node.sum()),n_cycles_kept=len(b),
                n_cycles_removed_vs_legacy=int(a.legacy_1node.sum())-len(b),
                affected=bool((a.legacy_1node & ~a[rule]).any()),
                completely_lost=bool(a.legacy_1node.any() and b.empty),
                cycle_min=b.Cycle_Index.min(),cycle_max=b.Cycle_Index.max()))
    return rows,cellrows


def bootstrap_effects(d,X,L,seed):
    # Kosul icinde fiziksel hucreler yeniden orneklenir; secilen hucrenin tum uygun cevrimleri korunur.
    groups=list(d.groupby('sample_int',sort=True).indices.items())
    nc=len(groups);p=X.shape[1];Sxx=np.zeros((nc,p,p));Sxy=np.zeros((nc,p));cids=[]
    y=d.Voltage_band.to_numpy(float)
    for k,(sid,pos) in enumerate(groups):
        Z=X[pos];Sxx[k]=Z.T@Z;Sxy[k]=Z.T@y[pos];cids.append(int(d.iloc[pos[0]].Condition_ID))
    rng=np.random.default_rng(seed);counts=np.zeros((N_BOOT,nc))
    cids=np.array(cids)
    for cid in np.unique(cids):
        idx=np.flatnonzero(cids==cid);m=len(idx)
        counts[:,idx]=rng.multinomial(m,np.full(m,1/m),size=N_BOOT)
    xx=(counts@Sxx.reshape(nc,-1)).reshape(N_BOOT,p,p);xy=counts@Sxy
    effects=np.full((N_BOOT,len(SPECS)),np.nan);invalid=0
    for b in range(N_BOOT):
        # Sahte katsayi ureten singular pseudoinverse fallback YOK.
        if np.linalg.matrix_rank(xx[b])<p:
            invalid+=1;continue
        try: beta=np.linalg.solve(xx[b],xy[b])
        except np.linalg.LinAlgError:
            invalid+=1;continue
        effects[b]=L@beta
    return effects,invalid


def fit_band(q,rule,band_index):
    d=q[q[rule]].copy().reset_index(drop=True)
    present=sorted(d.Condition_ID.unique());idx={c:i for i,c in enumerate(present)}
    counts=d.groupby('Condition_ID').sample_int.nunique().to_dict()
    cyc=d.groupby('Condition_ID').size().to_dict()
    L=np.full((27,len(present)+2),np.nan)
    for j,s in enumerate(SPECS):
        if s['lhs_condition'] in idx and s['rhs_condition'] in idx:
            L[j]=0
            L[j,idx[s['lhs_condition']]]=1;L[j,idx[s['rhs_condition']]]=-1
    fitted=None;boot=None;invalid=0;status='empty_band';diagnostics={}
    if len(d):
        X=np.column_stack([(d.Condition_ID.to_numpy()[:,None]==present).astype(float),d.age_c.to_numpy(),d.age_c.to_numpy()**2])
        rank=np.linalg.matrix_rank(X);p=X.shape[1];n=len(d);g=d.sample_int.nunique()
        status='ok' if rank==p and n>p and g>=2 else 'rank_or_cluster_deficiency'
        diagnostics=dict(n_cycles=n,n_cells=g,n_conditions=len(present),rank=rank,n_parameters=p,condition_number=float(np.linalg.cond(X)))
        if status=='ok':
            fitted=sm.OLS(d.Voltage_band.to_numpy(),X).fit(cov_type='cluster',
                cov_kwds={'groups':d.sample_int.to_numpy(),'use_correction':True},use_t=False)
            diagnostics.update(age_linear=fitted.params[-2],age_quadratic=fitted.params[-1],r_squared=fitted.rsquared)
            if rule in BOOT_RULES:
                print(f'  {rule}, {q.SoC_band.iloc[0]}: {N_BOOT} bootstrap...',flush=True)
                boot,invalid=bootstrap_effects(d,X,L,SEED+band_index*100+(0 if rule=='legacy_1node' else 1))
    rows=[]
    for j,s in enumerate(SPECS):
        l=s['lhs_condition'];r=s['rhs_condition'];nl=counts.get(l,0);nr=counts.get(r,0)
        rstatus='missing_condition_arm' if min(nl,nr)==0 else status
        row=dict(rule=rule,SoC_band=q.SoC_band.iloc[0],**s,n_cells_lhs=nl,n_cells_rhs=nr,
                 n_cycles_lhs=cyc.get(l,0),n_cycles_rhs=cyc.get(r,0),status=rstatus,
                 sparse_arm_lt5=min(nl,nr)<5,inference_eligible=False,
                 delta_age_adjusted_V=np.nan,delta_unadjusted_V=np.nan,SE_V=np.nan,
                 CI95_low_V=np.nan,CI95_high_V=np.nan,p_value=np.nan,
                 bootstrap_status='not_requested' if rule not in BOOT_RULES else 'model_unavailable',
                 bootstrap_valid=0,bootstrap_CI95_low_V=np.nan,bootstrap_CI95_high_V=np.nan,
                 bootstrap_sign_stability=np.nan,bootstrap_bucket_stability=np.nan)
        if rstatus=='ok':
            effect=float(L[j]@fitted.params)
            row['delta_age_adjusted_V']=effect
            row['delta_unadjusted_V']=float(d.loc[d.Condition_ID.eq(l),'Voltage_band'].mean()-d.loc[d.Condition_ID.eq(r),'Voltage_band'].mean())
            # Tek hucreli kolun belirsizligi fiziksel hucre bootstrap ile ogrenilemez.
            row['inference_eligible']=min(nl,nr)>=2
            if min(nl,nr)>=2:
                test=fitted.t_test(L[j]);ci=np.asarray(test.conf_int()).ravel()
                row.update(SE_V=float(np.asarray(test.sd).item()),CI95_low_V=float(ci[0]),
                    CI95_high_V=float(ci[1]),p_value=float(np.asarray(test.pvalue).item()))
            if boot is not None:
                vals=boot[:,j];vals=vals[np.isfinite(vals)];row['bootstrap_valid']=len(vals)
                bstatus='single_cell_arm' if min(nl,nr)<2 else 'insufficient_valid_replicates' if len(vals)<np.ceil(N_BOOT*MIN_BOOT_VALID_FRACTION) else 'ok'
                row['bootstrap_status']=bstatus
                if bstatus=='ok':
                    low,high=np.quantile(vals,[.025,.975]);row.update(bootstrap_CI95_low_V=low,
                        bootstrap_CI95_high_V=high,bootstrap_sign_stability=np.mean([sign(v)==sign(effect) for v in vals]),
                        bootstrap_bucket_stability=np.mean([bucket(v)==bucket(effect) for v in vals]))
        row['point_sign']=sign(row['delta_age_adjusted_V']);row['point_bucket']=bucket(row['delta_age_adjusted_V'])
        rows.append(row)
    diagnostics.update(rule=rule,SoC_band=q.SoC_band.iloc[0],status=status,bootstrap_invalid=invalid)
    return rows,diagnostics,boot


def run(blob,out,old_csv=None):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    d,before=load_cycles(blob)
    allq=[];coverage=[];cells=[];contrasts=[];diagnostics=[];draws={}
    with threadpool_limits(limits=1):
        for bi,(band,lo,hi) in enumerate(BANDS):
            q=band_data(d,band,lo,hi);allq.append(q)
            a,b=coverage_tables(q);coverage+=a;cells+=b
            for rule in RULES:
                print(f'Analiz: {band}, {rule}',flush=True)
                a,diag,boot=fit_band(q,rule,bi);contrasts+=a;diagnostics.append(diag)
                if boot is not None: draws[f'{rule}_band_{lo}_{hi}']=boot
    c=pd.DataFrame(contrasts)
    assert len(c)==4*81
    c['q_BH81']=np.nan
    # Her kurala ayri 81 planli test ailesi. Hesaplanamayan testler 1 ile yer tutar;
    # raporda p ve q degerleri NaN kalir; aile boyutu sessizce azaltilmaz.
    for rule in RULES:
        ii=c.index[c.rule.eq(rule)];v=c.loc[ii,'p_value'].to_numpy();ok=np.isfinite(v)
        adjusted=multipletests(np.where(ok,v,1.0),method='fdr_bh')[1]
        c.loc[ii,'q_BH81']=np.where(ok,adjusted,np.nan)
    for name in ('delta_age_adjusted_V','delta_unadjusted_V','SE_V','CI95_low_V','CI95_high_V','bootstrap_CI95_low_V','bootstrap_CI95_high_V'):
        c[name.replace('_V','_mV')]=1000*c[name]
    c['bootstrap_CI_excludes_zero']=pd.Series(pd.NA,index=c.index,dtype='boolean')
    bok=c.bootstrap_status.eq('ok')
    c.loc[bok,'bootstrap_CI_excludes_zero']=(c.loc[bok,'bootstrap_CI95_low_V'].gt(0)|c.loc[bok,'bootstrap_CI95_high_V'].lt(0)).to_numpy()
    baseline=c[c.rule.eq('legacy_1node')][['SoC_band','contrast_id','delta_age_adjusted_mV','point_sign','point_bucket','q_BH81','status','bootstrap_CI_excludes_zero']]
    compare=c.merge(baseline,on=['SoC_band','contrast_id'],suffixes=('','_legacy'),validate='many_to_one')
    compare['comparable']=np.isfinite(compare.delta_age_adjusted_mV)&np.isfinite(compare.delta_age_adjusted_mV_legacy)
    compare['change_mV']=compare.delta_age_adjusted_mV-compare.delta_age_adjusted_mV_legacy
    compare['absolute_magnitude_change_mV']=abs(compare.delta_age_adjusted_mV)-abs(compare.delta_age_adjusted_mV_legacy)
    for dest,col in [('sign_changed','point_sign'),('bucket_changed','point_bucket')]:
        compare[dest]=pd.Series(pd.NA,index=compare.index,dtype='boolean')
        compare.loc[compare.comparable,dest]=(compare.loc[compare.comparable,col]!=compare.loc[compare.comparable,col+'_legacy']).to_numpy()
    compare['FDR_decision_changed']=pd.Series(pd.NA,index=compare.index,dtype='boolean')
    valid=compare.q_BH81.notna()&compare.q_BH81_legacy.notna()
    compare.loc[valid,'FDR_decision_changed']=(compare.loc[valid,'q_BH81'].lt(.05)!=compare.loc[valid,'q_BH81_legacy'].lt(.05)).to_numpy()
    compare['bootstrap_CI_decision_changed']=pd.Series(pd.NA,index=compare.index,dtype='boolean')
    bv=compare.bootstrap_CI_excludes_zero.notna()&compare.bootstrap_CI_excludes_zero_legacy.notna()
    compare.loc[bv,'bootstrap_CI_decision_changed']=(compare.loc[bv,'bootstrap_CI_excludes_zero']!=compare.loc[bv,'bootstrap_CI_excludes_zero_legacy']).to_numpy()
    c.to_csv(out/'contrasts_all_rules.csv',index=False)
    compare.to_csv(out/'contrast_changes_vs_legacy.csv',index=False)
    compare[~compare.rule.eq('legacy_1node') & (~compare.comparable | compare.sign_changed.fillna(False) | compare.bucket_changed.fillna(False))].to_csv(out/'changed_or_unavailable_contrasts.csv',index=False)
    pd.DataFrame(coverage).merge(META,on='Condition_ID').to_csv(out/'coverage_by_condition_band_rule.csv',index=False)
    cell=pd.DataFrame(cells);cell.to_csv(out/'coverage_by_cell_band_rule.csv',index=False)
    cell.groupby(['rule','SoC_band']).agg(n_cycles_input=('n_cycles_input','sum'),n_cycles_legacy=('n_cycles_legacy','sum'),n_cycles_kept=('n_cycles_kept','sum'),n_cycles_removed=('n_cycles_removed_vs_legacy','sum'),n_cells_affected=('affected','sum'),n_cells_lost=('completely_lost','sum')).reset_index().to_csv(out/'coverage_summary.csv',index=False)
    q=pd.concat(allq,ignore_index=True);q.to_csv(out/'cycle_band_coverage_audit.csv.gz',index=False,compression='gzip')
    q[q.coverage_80].to_csv(out/'primary80_cycle_band_data.csv.gz',index=False,compression='gzip')
    pd.DataFrame(diagnostics).to_csv(out/'model_diagnostics.csv',index=False)
    pd.DataFrame(SPECS).to_csv(out/'contrast_specifications.csv',index=False)
    np.savez_compressed(out/'bootstrap_draws.npz',**draws)
    summary=compare.groupby('rule').agg(n_planned=('contrast_id','size'),n_comparable=('comparable','sum'),n_sign_changes=('sign_changed','sum'),n_bucket_changes=('bucket_changed','sum'),n_FDR_changes=('FDR_decision_changed','sum'),max_abs_change_mV=('change_mV',lambda x:abs(x).max())).reset_index()
    summary.to_csv(out/'contrast_change_summary.csv',index=False)
    if old_csv is not None:
        old=pd.read_csv(io.BytesIO(old_csv));keys=['SoC_band','contrast_id']
        if not set(keys+['delta_age_adjusted_V']).issubset(old) or old.duplicated(keys).any():
            raise ValueError('Opsiyonel eski R7 CSV semasi veya kontrast anahtarlari gecersiz.')
        check=c[c.rule.eq('legacy_1node')].merge(old[keys+['delta_age_adjusted_V']],on=keys,suffixes=('_new','_old'),how='outer',indicator=True)
        check['difference_mV']=(check.delta_age_adjusted_V_new-check.delta_age_adjusted_V_old)*1000
        check.to_csv(out/'optional_old_R7_reproduction_check.csv',index=False)
        print('Eski R7 nokta tahminleriyle en buyuk fark (mV):',check.difference_mV.abs().max())
    cfg=dict(source_sha256=hashlib.sha256(blob).hexdigest(),timestamp_utc=datetime.now(timezone.utc).isoformat(),
        rules=RULES,bands=BANDS,bootstrap_replicates=N_BOOT,seed=SEED,reference_cycle=REFERENCE_CYCLE,
        input_rows=before,nominal_cycles=len(d),nominal_cells=d.sample_int.nunique(),
        model='Voltage_band ~ 0 + C(Condition_ID) + age_c + age_c^2',
        covariance='physical-cell cluster CR1, normal reference',BH_family_size=81,
        bootstrap_rules=sorted(BOOT_RULES),python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,statsmodels=sm.__version__)
    (out/'run_config.json').write_text(json.dumps(cfg,indent=2),encoding='utf-8')
    (out/'README.txt').write_text(
        'BANT KAPSAMI REVIZYONU / tam cevrim gecmisi.\n'
        'Bantlar [0,20), [20,60), [60,90): 20,40,30 dugum; 90 dahil degildir.\n'
        'Ana %80 esikleri 16/32/24; %60:12/24/18; %100:20/40/30; legacy:1/1/1.\n'
        'Eksik dugumler doldurulmaz. Ortalama sadece sonlu dugumlerdendir.\n'
        'Ana kural sonuclara bakilarak secilmez. Kapsam esigi tam bant ortalamasini garanti etmez.\n'
        'R7 ortak yas modeli korunur; 108 merkezleme noktasidir, kontrastlar ortak yas terimlerinden bagimsizdir.\n'
        'Hucresi eksik kosul modelden cikar; bu kola ait kontrast NA kalir.\n'
        'Tek hucreli kol varsa nokta tahmini betimseldir; p, CI ve bootstrap belirsizligi raporlanmaz.\n'
        '2-4 hucreli kollar sparse_arm_lt5 olarak isaretlenir; belirsizlik temkinli yorumlanmalidir.\n'
        'CR1 global hucre-kume kovaryansidir; hucre katkilarini dengelemez.\n'
        'Her esik icin 81 planli kontrasta ayri BH uygulanir. Hesaplanamayan testler aile hesabinda p=1\n'
        'yer tutar; kendi p/q alanlari NA kalir. Farkli esikler tek bir 324 test ailesi diye birlestirilmez.\n'
        'Bootstrap legacy ve %80 icin kosul icinde hucre ornekleme; tum uygun cevrimler birlikte tasinir.\n'
        'Her bant ayri orneklenir. Bootstrap yuzdelik CI noktasaldir, eszamanli/FDR duzeltilmis degildir.\n'
        'Rank eksik bootstrap tekrarlarinda pseudoinverse kullanilmaz; gecerli tekrar orani <%95 ise CI verilmez.\n'
        'Pratik siniflar mutlak etki: <50 negligible, 50-100 small, 100-150 moderate, >=150 large (mV).\n'
        'Degisimler kapsam secimi ve tum modelin yeniden uyumunu birlikte yansitir; nedensel etki degildir.\n'
        'Bootstrap draw sutun sirasi contrast_specifications.csv satir sirasidir; degerler V biriminde.\n'
        'primary80_cycle_band_data.csv.gz sonraki yaslanma duyarliligi icin hazir girdidir.\n'
        'Opsiyonel eski R7 dosyasi yoksa arsivlenmis R7 sonucunun aynen yeniden uretildigi iddia edilmez.\n'
        'Eski R8 dosyasi karsilastirilmaz; legacy bootstrap ayni tasarimin yeni tohumla tekraridir.\n',encoding='utf-8')
    print('\n81 KONTRAST / KURAL KARSILASTIRMASI\n',summary.to_string(index=False))
    print('\nAna %80: hesaplanamayan veya tek hucreli kollar\n',c[c.rule.eq('coverage_80')&(~c.status.eq('ok')|~c.inference_eligible)][['SoC_band','contrast_id','n_cells_lhs','n_cells_rhs','status']].to_string(index=False))
    return c


def main():
    from google.colab import files
    print('global_cycle_profiles.pkl dosyasini secin. Isterseniz eski R7 kontrast CSV dosyasini da ekleyin.')
    uploaded=files.upload()
    pkls=[(n,b) for n,b in uploaded.items() if n.lower().endswith('.pkl')]
    csvs=[(n,b) for n,b in uploaded.items() if n.lower().endswith('.csv')]
    if len(pkls)!=1 or len(csvs)>1 or len(uploaded)!=len(pkls)+len(csvs):
        raise ValueError('Bir PKL ve en fazla bir eski R7 CSV yukleyin.')
    out=Path(tempfile.mkdtemp(prefix='SoC_band_coverage_'))
    run(pkls[0][1],out,csvs[0][1] if csvs else None)
    archive=shutil.make_archive(str(out),'zip',root_dir=out)
    print('\nTamamlandi:',archive);files.download(archive)


# Original auto-run omitted: the runner below supplies audited local inputs.

# ---------- LOCAL, NONDESTRUCTIVE STAGE RUNNER ----------
source = require_path(INPUT_CYCLES)
out = stage_dir('01_band_coverage')
print('SOURCE R2.2:', source)
run(source.read_bytes(), out)
import pandas as pd
x = pd.read_csv(out/'coverage_summary.csv')
print('COVERAGE 80:', x[x['rule'].eq('coverage_80')].to_string(index=False))
y = pd.read_csv(out/'contrasts_all_rules.csv')
assert len(y[y.rule.eq('coverage_80')]) == 81, 'Expected 81 planned primary contrasts'
print('Primary contrasts:',len(y[y.rule.eq('coverage_80')]),'eligible:',int(y[y.rule.eq('coverage_80')]['inference_eligible'].sum()))
stage_zip('01_band_coverage','SoC_band_coverage_')


## 02 — Four-model cycle-history and physical-cell weighting sensitivity

Original `ConditionvCodes(2).py` lines **30,394–30,701**. Run after the prior required stage.

In [ ]:
# COLAB: Bu dosyanin tamamini TEK hucreye yapistirip calistirin.
# Manuel girdi: SoC_band_coverage_ts2_1sbp.zip (3. adimin sonuc ZIP'i).
# Ham veri gerekmez. Ana %80 kapsam kurali degistirilmez.
import io, json, hashlib, tempfile, shutil, platform
from pathlib import Path
from zipfile import ZipFile
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from threadpoolctl import threadpool_limits

N_BOOT=2000
SEED=20260928
MIN_VALID_FRACTION=.95
REF=108
LOCAL_RADIUS=10
BANDS=('0-20','20-60','60-90')
MODELS=('reference','condition_slopes','cell_balanced','window_52_132')
TIMES=(52,108,132)
# condition_slopes icin destek kuralimiz:
# her kolda >=2 fiziksel hucrenin kendi cevrim min/max araligi sorgu cevrimini kapsar.
# +/-10 cevrimdeki hucre ve gozlem sayilari ayrica tani amaciyla raporlanir.


def read_member(z,name):
    hits=[x for x in z.namelist() if x==name or x.endswith('/'+name)]
    if len(hits)!=1: raise ValueError(f'Arsivde tek bir {name} gerekli: {hits}')
    return z.read(hits[0])


def read_inputs(blob):
    with ZipFile(io.BytesIO(blob)) as z:
        d=pd.read_csv(io.BytesIO(read_member(z,'primary80_cycle_band_data.csv.gz')),compression='gzip')
        specs=pd.read_csv(io.BytesIO(read_member(z,'contrast_specifications.csv')))
        previous=pd.read_csv(io.BytesIO(read_member(z,'contrasts_all_rules.csv')))
    cols=['sample_int','Condition_ID','Cycle_Index','SoC_band','Voltage_band','n_valid','n_band_nodes']
    if not set(cols).issubset(d): raise ValueError('primary80 verisinin semasi eksik.')
    for c in cols:
        if c!='SoC_band': d[c]=pd.to_numeric(d[c],errors='raise')
    if not np.isfinite(d[[c for c in cols if c!='SoC_band']].to_numpy()).all():
        raise ValueError('Ana %80 verisinde sonlu olmayan sayisal deger var.')
    for c in ['sample_int','Condition_ID','Cycle_Index','n_valid','n_band_nodes']:
        if not np.allclose(d[c],np.round(d[c]),rtol=0,atol=1e-8): raise ValueError(f'{c} tamsayi olmali.')
        d[c]=np.round(d[c]).astype(int)
    if set(d.SoC_band)!=set(BANDS) or not d.Condition_ID.between(1,24).all():
        raise ValueError('Bant/kosul degerleri gecersiz.')
    if d.duplicated(['sample_int','Condition_ID','Cycle_Index','SoC_band']).any():
        raise ValueError('Yinelenen cevrim-bant satiri var.')
    if (d.groupby('sample_int').Condition_ID.nunique()>1).any(): raise ValueError('Hucre birden cok kosulda.')
    expected=d.SoC_band.map({'0-20':20,'20-60':40,'60-90':30})
    if not (d.n_band_nodes.eq(expected)&d.n_valid.ge(np.ceil(.8*expected))&d.n_valid.le(expected)).all():
        raise ValueError('%80 kapsam kurali veya bant boyutu uyusmuyor.')
    needed=['contrast_id','contrast_family','lhs_condition','rhs_condition']
    if not set(needed).issubset(specs) or len(specs)!=27 or specs.contrast_id.duplicated().any():
        raise ValueError('27 benzersiz kontrast tanimi gerekli.')
    for col in ['lhs_condition','rhs_condition']:
        if not specs[col].isin(range(1,25)).all(): raise ValueError('Kontrast kosulu gecersiz.')
    previous=previous[previous.rule.eq('coverage_80')]
    if len(previous)!=81 or previous.duplicated(['SoC_band','contrast_id']).any():
        raise ValueError('3. adimin 81 adet %80 kontrasti gerekli.')
    d['age_c']=(d.Cycle_Index-REF)/100.0
    return d.reset_index(drop=True),specs,previous


def sign(v):
    return 'NA' if not np.isfinite(v) else 'positive' if v>1e-12 else 'negative' if v< -1e-12 else 'zero'


def bucket(v):
    return 'NA' if not np.isfinite(v) else 'negligible' if abs(v)<.05 else 'small' if abs(v)<.10 else 'moderate' if abs(v)<.15 else 'large'


def support(d,t):
    rows={}
    for cid in range(1,25):
        g=d[d.Condition_ID.eq(cid)]
        per=g.groupby('sample_int').Cycle_Index.agg(['min','max'])
        local=g[g.Cycle_Index.between(t-LOCAL_RADIUS,t+LOCAL_RADIUS)]
        rows[cid]=dict(n_cells=g.sample_int.nunique(),n_cycles=len(g),
            n_distinct_cycles=g.Cycle_Index.nunique(),cycle_min=g.Cycle_Index.min(),cycle_max=g.Cycle_Index.max(),
            n_cells_bracketing=int((per['min'].le(t)&per['max'].ge(t)).sum()),
            n_local_cells=local.sample_int.nunique(),n_local_cycles=len(local),
            inside_condition_range=bool(len(g) and g.Cycle_Index.min()<=t<=g.Cycle_Index.max()))
    return rows


def design(d,name):
    levels=sorted(d.Condition_ID.unique());k=len(levels)
    D=(d.Condition_ID.to_numpy()[:,None]==levels).astype(float)
    age=d.age_c.to_numpy()
    X=np.column_stack([D,D*age[:,None],age**2]) if name=='condition_slopes' else np.column_stack([D,age,age**2])
    w=(1/d.groupby('sample_int').sample_int.transform('size')).to_numpy() if name=='cell_balanced' else np.ones(len(d))
    return X,w,levels


def contrast_matrix(specs,levels,p,t,name):
    idx={cid:i for i,cid in enumerate(levels)};L=np.full((len(specs),p),np.nan)
    for j,s in enumerate(specs.itertuples()):
        if s.lhs_condition not in idx or s.rhs_condition not in idx: continue
        l,r=idx[s.lhs_condition],idx[s.rhs_condition];L[j]=0;L[j,l]=1;L[j,r]=-1
        if name=='condition_slopes':L[j,len(levels)+l]=(t-REF)/100;L[j,len(levels)+r]=-(t-REF)/100
    return L


def fit_one(d,name,specs,rng_seed):
    times=TIMES if name=='condition_slopes' else (108,)
    X,w,levels=design(d,name);p=X.shape[1];n=len(d)
    rank=np.linalg.matrix_rank(X*np.sqrt(w[:,None])) if n else 0
    status='ok' if n>p and rank==p and d.sample_int.nunique()>=2 else 'rank_or_sample_deficiency'
    diag=dict(model=name,n_cycles=n,n_cells=d.sample_int.nunique(),n_conditions=len(levels),
        n_parameters=p,rank=rank,status=status,condition_number=float(np.linalg.cond(X*np.sqrt(w[:,None]))) if n else np.nan,
        bootstrap_requested=N_BOOT,bootstrap_invalid=0)
    matrices={t:contrast_matrix(specs,levels,p,t,name) for t in times}
    supp={t:support(d,t) for t in times}
    effects={t:np.full((N_BOOT,len(specs)),np.nan) for t in times}
    unadjusted_boot=np.full((N_BOOT,len(specs)),np.nan)
    fit=None;raw=np.full(len(specs),np.nan);coeff=[]
    if status=='ok':
        fit=sm.WLS(d.Voltage_band.to_numpy(),X,weights=w).fit(cov_type='cluster',
            cov_kwds={'groups':d.sample_int.to_numpy(),'use_correction':True},use_t=False)
        diag['R_squared']=fit.rsquared
        names=[f'condition_{c}_intercept' for c in levels]
        names+=([f'condition_{c}_slope' for c in levels] if name=='condition_slopes' else ['common_linear'])+['common_quadratic']
        coeff=[dict(term=term,estimate_V=float(val)) for term,val in zip(names,fit.params)]
        groups=list(d.groupby('sample_int',sort=True).indices.items());nc=len(groups)
        sxx=np.zeros((nc,p,p));sxy=np.zeros((nc,p));cids=[]
        y=d.Voltage_band.to_numpy();k=len(levels)
        for j,(sid,pos) in enumerate(groups):
            Z=X[pos];sxx[j]=Z.T@(w[pos,None]*Z);sxy[j]=Z.T@(w[pos]*y[pos]);cids.append(int(d.iloc[pos[0]].Condition_ID))
        totalxx=sxx.sum(axis=0);totalxy=sxy.sum(axis=0)
        raw=matrices[108][:,:k]@(totalxy[:k]/np.diag(totalxx)[:k])
        rng=np.random.default_rng(rng_seed);counts=np.zeros((N_BOOT,nc));cids=np.asarray(cids)
        for cid in levels:
            ii=np.flatnonzero(cids==cid);m=len(ii)
            counts[:,ii]=rng.multinomial(m,np.full(m,1/m),size=N_BOOT)
        # Her bootstrap kopyasi kendi hucre agirligini korur: balanced toplam agirlik/kopya=1.
        bxx=(counts@sxx.reshape(nc,-1)).reshape(N_BOOT,p,p);bxy=counts@sxy
        for b in range(N_BOOT):
            if np.linalg.matrix_rank(bxx[b])<p:
                diag['bootstrap_invalid']+=1;continue
            try:beta=np.linalg.solve(bxx[b],bxy[b])
            except np.linalg.LinAlgError:
                diag['bootstrap_invalid']+=1;continue
            for t in times:effects[t][b]=matrices[t]@beta
            unadjusted_boot[b]=matrices[108][:,:k]@(bxy[b,:k]/np.diag(bxx[b])[:k])
    rows=[];support_rows=[]
    for t in times:
        for cid,sp in supp[t].items():support_rows.append(dict(reference_cycle=t,Condition_ID=cid,**sp))
        for j,s in enumerate(specs.to_dict('records')):
            a,b=supp[t][s['lhs_condition']],supp[t][s['rhs_condition']]
            missing=min(a['n_cells'],b['n_cells'])==0
            inside=a['inside_condition_range'] and b['inside_condition_range']
            replicated=min(a['n_cells_bracketing'],b['n_cells_bracketing'])>=2
            reason='missing_arm' if missing else status if status!='ok' else 'single_cell_arm' if min(a['n_cells'],b['n_cells'])<2 else 'reference_outside_condition_range' if name=='condition_slopes' and not inside else 'insufficient_bracketing_cells' if name=='condition_slopes' and not replicated else 'ok'
            r=dict(**s,reference_cycle=t,status=reason,inference_eligible=reason=='ok',
                support_inside_both_ranges=inside,support_two_bracketing_cells_each=replicated,
                n_cells_lhs=a['n_cells'],n_cells_rhs=b['n_cells'],n_cycles_lhs=a['n_cycles'],n_cycles_rhs=b['n_cycles'],
                n_bracketing_cells_lhs=a['n_cells_bracketing'],n_bracketing_cells_rhs=b['n_cells_bracketing'],
                n_local_cells_lhs=a['n_local_cells'],n_local_cells_rhs=b['n_local_cells'],
                sparse_arm_lt5=min(a['n_cells'],b['n_cells'])<5,
                delta_adjusted_V=np.nan,delta_unadjusted_V=np.nan,absolute_shrinkage_V=np.nan,
                p_value=np.nan,cluster_CI_low_V=np.nan,cluster_CI_high_V=np.nan,
                bootstrap_CI_low_V=np.nan,bootstrap_CI_high_V=np.nan,bootstrap_sign_stability=np.nan,
                shrinkage_CI_low_V=np.nan,shrinkage_CI_high_V=np.nan,bootstrap_shrinkage_fraction=np.nan,
                bootstrap_valid=0,bootstrap_status='ineligible')
            if not missing and fit is not None:
                L=matrices[t][j];point=float(L@fit.params)
                r.update(delta_adjusted_V=point,delta_unadjusted_V=float(raw[j]),absolute_shrinkage_V=abs(raw[j])-abs(point))
                if reason=='ok':
                    test=fit.t_test(L);ci=np.asarray(test.conf_int()).ravel()
                    r.update(p_value=float(np.asarray(test.pvalue).item()),cluster_CI_low_V=float(ci[0]),cluster_CI_high_V=float(ci[1]))
                    v=effects[t][:,j];u=unadjusted_boot[:,j];valid=np.isfinite(v)&np.isfinite(u);v=v[valid];u=u[valid]
                    r['bootstrap_valid']=len(v)
                    r['bootstrap_status']='ok' if len(v)>=np.ceil(N_BOOT*MIN_VALID_FRACTION) else 'too_few_valid_replicates'
                    if r['bootstrap_status']=='ok':
                        lo,hi=np.quantile(v,[.025,.975]);sh=abs(u)-abs(v);sl,shigh=np.quantile(sh,[.025,.975])
                        r.update(bootstrap_CI_low_V=lo,bootstrap_CI_high_V=hi,
                            bootstrap_sign_stability=np.mean([sign(x)==sign(point) for x in v]),
                            shrinkage_CI_low_V=sl,shrinkage_CI_high_V=shigh,bootstrap_shrinkage_fraction=np.mean(sh>0))
            r['point_sign']=sign(r['delta_adjusted_V']);r['point_bucket']=bucket(r['delta_adjusted_V'])
            rows.append(r)
    return rows,diag,coeff,support_rows,effects


def run(blob,out):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    d,specs,previous=read_inputs(blob)
    rows=[];diagnostics=[];coef=[];sup=[];draws={}
    with threadpool_limits(limits=1):
        for bi,band in enumerate(BANDS):
            base=d[d.SoC_band.eq(band)].copy()
            for mi,name in enumerate(MODELS):
                use=base[base.Cycle_Index.between(52,132)].copy() if name=='window_52_132' else base.copy()
                use=use.reset_index(drop=True)
                print(f'{band} / {name}: {len(use):,} cevrim, {use.sample_int.nunique()} hucre; {N_BOOT} bootstrap',flush=True)
                rr,dd,cc,ss,ee=fit_one(use,name,specs,SEED+bi*100+mi)
                rows.extend(dict(model=name,SoC_band=band,**r) for r in rr)
                diagnostics.append(dict(SoC_band=band,**dd))
                coef.extend(dict(model=name,SoC_band=band,**r) for r in cc)
                sup.extend(dict(model=name,SoC_band=band,**r) for r in ss)
                for t,v in ee.items():draws[f'{name}_band{bi}_cycle{t}']=v
    c=pd.DataFrame(rows)
    assert len(c)==486 # 81*(reference + 3 slope times + balanced + window)
    c['q_BH81']=np.nan
    for _,g in c.groupby(['model','reference_cycle']):
        assert len(g)==81
        p=g.p_value.to_numpy();finite=np.isfinite(p)
        q=multipletests(np.where(finite,p,1),method='fdr_bh')[1]
        c.loc[g.index,'q_BH81']=np.where(finite,q,np.nan)
    vcols=[x for x in c if x.endswith('_V')]
    for col in vcols:c[col[:-2]+'_mV']=c[col]*1000
    c['bootstrap_excludes_zero']=pd.Series(pd.NA,index=c.index,dtype='boolean')
    ok=c.bootstrap_status.eq('ok')
    c.loc[ok,'bootstrap_excludes_zero']=(c.loc[ok,'bootstrap_CI_low_V'].gt(0)|c.loc[ok,'bootstrap_CI_high_V'].lt(0)).to_numpy()
    # Asama 3 modeline sayisal geri-dogrulama.
    check=c[c.model.eq('reference')].merge(previous[['SoC_band','contrast_id','delta_age_adjusted_V']],on=['SoC_band','contrast_id'],validate='one_to_one')
    check['difference_mV']=(check.delta_adjusted_V-check.delta_age_adjusted_V)*1000
    check.to_csv(out/'reference_reproduction_check.csv',index=False)
    if len(check)!=81 or not np.allclose(check.delta_adjusted_V,check.delta_age_adjusted_V,rtol=0,atol=1e-8,equal_nan=True):
        raise RuntimeError('Referans 3. adimla uyusmuyor; sonuclari kullanmayin. Kontrol CSVsine bakin.')
    ref=c[c.model.eq('reference')][['SoC_band','contrast_id','delta_adjusted_mV','point_sign','point_bucket','inference_eligible']]
    comparison=c.merge(ref,on=['SoC_band','contrast_id'],suffixes=('','_reference'),validate='many_to_one')
    comparison['change_vs_reference_mV']=comparison.delta_adjusted_mV-comparison.delta_adjusted_mV_reference
    comparison['primary_comparable']=comparison.inference_eligible & comparison.inference_eligible_reference
    for target,col in [('sign_changed','point_sign'),('bucket_changed','point_bucket')]:
        comparison[target]=pd.Series(pd.NA,index=comparison.index,dtype='boolean')
        mask=comparison.primary_comparable
        comparison.loc[mask,target]=(comparison.loc[mask,col]!=comparison.loc[mask,col+'_reference']).to_numpy()
    # Ayni destekli kontrast kumesi: 108'de dort analiz de uygun olmali.
    at108=comparison[comparison.reference_cycle.eq(108)].copy()
    common=at108.groupby(['SoC_band','contrast_id']).agg(all_eligible=('primary_comparable','all'),n_models=('model','nunique'))
    common['common_four_models']=common.all_eligible & common.n_models.eq(4)
    comparison=comparison.merge(common[['common_four_models']],on=['SoC_band','contrast_id'],validate='many_to_one')
    c.to_csv(out/'all_model_contrasts.csv',index=False)
    comparison.to_csv(out/'model_changes_vs_reference.csv',index=False)
    comparison[(comparison.reference_cycle==108)&comparison.common_four_models].to_csv(out/'common_supported_contrasts_at108.csv',index=False)
    c[c.contrast_family.eq('temperature_60_minus_25')].to_csv(out/'thermal_adjustment_and_shrinkage.csv',index=False)
    c[c.contrast_family.isin(['rate_2C_minus_1C','rate_0.7C_minus_2C'])].to_csv(out/'aggressive_rate_robustness.csv',index=False)
    c[~c.inference_eligible].to_csv(out/'unsupported_or_descriptive_contrasts.csv',index=False)
    pd.DataFrame(sup).to_csv(out/'condition_cycle_support.csv',index=False)
    pd.DataFrame(diagnostics).to_csv(out/'model_diagnostics.csv',index=False)
    pd.DataFrame(coef).to_csv(out/'model_coefficients.csv',index=False)
    np.savez_compressed(out/'bootstrap_contrast_draws.npz',**draws)
    specs.to_csv(out/'contrast_specifications.csv',index=False)
    summary=[]
    for (name,t),g in comparison.groupby(['model','reference_cycle']):
        good=g[g.primary_comparable]
        summary.append(dict(model=name,reference_cycle=t,n_planned=len(g),n_eligible=int(g.inference_eligible.sum()),
            n_comparable=len(good),sign_changes=int(good.sign_changed.sum()),bucket_changes=int(good.bucket_changed.sum()),
            max_abs_change_mV=good.change_vs_reference_mV.abs().max(),
            n_bootstrap_supported=int(g.bootstrap_excludes_zero.fillna(False).sum())))
    pd.DataFrame(summary).to_csv(out/'model_comparison_summary.csv',index=False)
    cfg=dict(source_sha256=hashlib.sha256(blob).hexdigest(),timestamp_utc=datetime.now(timezone.utc).isoformat(),
        bootstrap_replicates=N_BOOT,seed=SEED,reference_cycle=REF,local_radius=LOCAL_RADIUS,
        slope_times=TIMES,min_bracketing_cells_per_arm=2,min_valid_bootstrap_fraction=MIN_VALID_FRACTION,
        models=MODELS,python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,statsmodels=sm.__version__,
        reference_max_difference_mV=float(check.difference_mV.abs().max()))
    (out/'run_config.json').write_text(json.dumps(cfg,indent=2),encoding='utf-8')
    (out/'README.txt').write_text(
        'ASAMA 4: Sabit %80 bant kapsamli veride yaslanma modeli duyarliligi.\n'
        'reference: kosul sabitleri + ortak age + age^2, age=(Cycle_Index-108)/100.\n'
        'condition_slopes: kosul sabitleri + kosula ozgu dogrusal egimler + ortak age^2.\n'
        'cell_balanced: referans model, her satir agirligi=1/(ilgili hucrenin bu banttaki uygun cevrim sayisi).\n'
        'window_52_132: %80 uygun verinin 52..132 dahil alt kumesi, referans model.\n'
        'Tum modellerde fiziksel hucre CR1 kovaryansi ve normal referans dagilimi kullanilir.\n'
        'Duzeltilmemis fark, ayni modelin veri ve agirliklariyla kosul ortalamalarinin farkidir.\n'
        'absolute_shrinkage = abs(unadjusted)-abs(adjusted); pozitif deger mutlak kuculmedir.\n'
        'Egimli modelde unadjusted tum uygun cevrim gecmisi ortalamasidir; t cevrimine yerel ortalama degildir.\n'
        'Bootstrap kuculme araliklari ayni tekrar icindeki duzeltilmis/duzeltilmemis tahminlerden uretilir.\n'
        'Her kosul icinde hucreler yeniden orneklenir; secilen hucrenin tum uygun cevrimleri tasinir.\n'
        'Balanced bootstrap: her secilmis hucre kopyasinin toplam agirligi 1 olur; coklu secim bu agirligi carpandir.\n'
        'Tek hucreli kol: nokta betimsel; p/CI yok. Eksik kol: nokta da NA.\n'
        'Egimli model: her kolda en az iki hucrenin kendi min/max cevrim araligi t yi kapsamalidir.\n'
        'Bu destek kurali sonuclardan once sabitlenmistir; yerel +/-10 hucre sayilari ek tani olup ikinci esik degildir.\n'
        'Destek disi egimli nokta tahminleri denetim icin saklanir, primary_comparable=False; ana sonuca katilmaz.\n'
        'Ortak egimli modelin kontrasti t den bagimsizdir; support bilgisi raporlanir fakat t-destegi kontrasta ek kosul yapilmaz.\n'
        '52,108,132 egimli sonuclari ayri 81-test BH aileleridir. 108 ana, 52/132 kesifsel duyarliliktir.\n'
        'Eksik/uygunsuz testler BH hesabinda p=1 yer tutar; raporda q NA kalir. Modeller arasi ortak FDR iddiasi yoktur.\n'
        'Bootstrap CI yuzdelik ve noktasaldir; coklu karsilastirma duzeltilmis CI degildir.\n'
        'Rank eksik model/tekrarda pseudoinverse veya model degistirme yoktur. Gecerli tekrar <%95 ise CI verilmez.\n'
        'NPZ draw sutun sirasi contrast_specifications.csv; birim V. Destek disi ham drawlar ana CI icin kullanilmaz.\n'
        'common_supported_contrasts_at108 dort modelde de uygun ayni kontrastlarin karsilastirmasidir.\n'
        'Pencere analizi gozlem kumesini; balanced analiz hedeflenen agirliklandirmayi degistirir. Farklar salt model bicimine baglanamaz.\n'
        'Condition5 tek hucre/iki cevrimle temsil edildigi icin egimi guvenilir grup etkisi olarak yorumlanmaz.\n'
        'Yas etkisi cevrim gecmisi iliskisidir; elektrokimyasal yaslanmanin nedensel etkisi degildir.\n',encoding='utf-8')
    print('\nREFERANS KONTROLU: en buyuk fark (mV) =',cfg['reference_max_difference_mV'])
    print('\nMODEL KARSILASTIRMASI\n',pd.DataFrame(summary).to_string(index=False))
    print('\nTAMAM: Sonuclari model, destek ve bootstrap durumuyla birlikte okuyun.')
    return c,comparison


def main():
    from google.colab import files
    print('3. adimin sonuc ZIP dosyasini yukleyin: SoC_band_coverage_ts2_1sbp.zip')
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError('Tek ZIP yukleyin.')
    name,blob=next(iter(uploaded.items()))
    if not name.lower().endswith('.zip'):raise ValueError('ZIP dosyasi gerekli.')
    out=Path(tempfile.mkdtemp(prefix='SoC_age_sensitivity_'))
    run(blob,out)
    archive=shutil.make_archive(str(out),'zip',root_dir=out)
    print('\nSonuc ZIP:',archive);files.download(archive)


# Original auto-run omitted: the runner below supplies audited local inputs.

# ---------- LOCAL, NONDESTRUCTIVE STAGE RUNNER ----------
path = require_path('/content/SoC_band_coverage_ConditionvCodes2.zip')
out = stage_dir('02_age_sensitivity')
print('SOURCE COVERAGE:',path)
run(path.read_bytes(),out)
import pandas as pd
c = pd.read_csv(out/'all_model_contrasts.csv')
print('AGE CONTRAST ROWS:',len(c))
print('REPRODUCTION QA:')
print(pd.read_csv(out/'reference_reproduction_check.csv').head(8).to_string(index=False))
stage_zip('02_age_sensitivity','SoC_age_sensitivity_')


## 03 — Unified full-history and 52–132 LUT + LOCO/hold-outs

Original `ConditionvCodes(2).py` lines **29,446–29,796**. Run after the prior required stage.

In [ ]:
# GOOGLE COLAB: Bu dosyanin TAMAMINI tek hucreye yapistirip calistirin.
# Girdiler: cell_profiles_nominal_concordant.csv
#           broad_52_132_cell_profiles.csv
# Processed_Outputv3 ham dosyalari bu asamanin girdisi DEGILDIR.
import io, json, hashlib, warnings, shutil, tempfile, platform
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import sklearn
from sklearn.isotonic import IsotonicRegression

GRID = np.arange(101)
MIN_EVAL_POINTS = 5
FILES = {
    'full_history': 'cell_profiles_nominal_concordant.csv',
    'broad_52_132': 'broad_52_132_cell_profiles.csv',
}
METHODS = ('unified', 'legacy_R5')
KEYS = ['analysis', 'validation', 'Condition_ID', 'sample_int']
META = pd.DataFrame([
    (1 + 6*t + 2*r + k, T, cutoff, rate)
    for t, T in enumerate((10, 25, 45, 60))
    for r, rate in enumerate((0.7, 1.0, 2.0))
    for k, cutoff in enumerate(('C/5', 'C/40'))
], columns=['Condition_ID', 'Temperature', 'Rate', 'Charge_Rate'])
LOOKUP = {(r.Temperature, r.Rate, r.Charge_Rate): r.Condition_ID
          for r in META.itertuples()}


def load_profiles(blob, name):
    d = pd.read_csv(io.BytesIO(blob))
    d.columns = d.columns.str.strip()
    if 'V_cell_median' in d:
        if 'Voltage' in d:
            raise ValueError(f'{name}: iki voltaj sutunu var; belirsiz girdi.')
        d = d.rename(columns={'V_cell_median': 'Voltage'})
    required = ['Condition_ID', 'sample_int', 'SoC', 'Voltage']
    if not set(required).issubset(d.columns):
        raise ValueError(f'{name}: gerekli sutunlar: {required}')
    for c in required:
        d[c] = pd.to_numeric(d[c], errors='raise')
    for c in required[:3]:
        a = d[c].to_numpy(float)
        if not np.isfinite(a).all() or not np.allclose(a, np.round(a), rtol=0, atol=1e-8):
            raise ValueError(f'{name}: {c} tam sayi ve sonlu olmali.')
        d[c] = np.round(a).astype(int)
    if np.isinf(d.Voltage).any():
        raise ValueError(f'{name}: sonsuz voltaj degeri var.')
    if not d.Condition_ID.between(1, 24).all() or not d.SoC.between(0, 100).all():
        raise ValueError(f'{name}: kosul veya SoC araligi hatali.')
    if d.duplicated(required[:3]).any():
        raise ValueError(f'{name}: yinelenen hucre-kosul-SoC; sessizce birlestirilmedi.')
    if (d.groupby('sample_int').Condition_ID.nunique() > 1).any():
        raise ValueError(f'{name}: bir fiziksel hucre birden cok kosulda.')
    for (cid, sid), g in d.groupby(['Condition_ID', 'sample_int']):
        allowed = range(8*(cid-1)+1, 8*cid+1) if cid <= 22 else (
            range(177, 180) if cid == 23 else range(185, 188))
        if sid not in allowed:
            raise ValueError(f'{name}: nominal harita uyusmazligi: kosul={cid}, hucre={sid}')
        if set(g.SoC) != set(GRID):
            raise ValueError(f'{name}: hucre {sid}: 0..100 satirlari eksik. '
                             'Kirpilmamis ara ciktiyi yukleyin; eksik voltajlar NaN olabilir.')
    if name == 'full_history' and set(d.Condition_ID) != set(range(1, 25)):
        raise ValueError('Tam gecmis dosyasinda 24 kosulun tamami bulunmali.')
    # Varsa faktor bilgileri resmi kosul haritasiyla karsilastirilir.
    for c in ('Temperature', 'Rate', 'Charge_Rate'):
        if c not in d:
            continue
        expected = d.Condition_ID.map(META.set_index('Condition_ID')[c])
        if c == 'Rate':
            actual = d[c].astype(str).str.replace(' ', '', regex=False).str.upper()
            ok = actual.eq(expected)
        else:
            actual = d[c].astype(str).str.replace('°', '', regex=False)
            actual = actual.str.replace('º', '', regex=False).str.replace('C', '', regex=False)
            actual = pd.to_numeric(actual.str.strip(), errors='coerce')
            ok = np.isclose(actual, expected, rtol=0, atol=1e-8)
        if not np.asarray(ok).all():
            raise ValueError(f'{name}: {c} metadata uyusmazligi.')
    return d[required].merge(META, on='Condition_ID', validate='many_to_one')


def build_lut(A, method='unified'):
    """A: benzersiz egitim hucreleri x 101 SoC dugumu. Test verisi almaz.
    unified: >=1 hucre/dugum, destek agirlikli, 0..100'de fit,
    >=2 destekli fit dugumu; sifir destekli cikti kesinlikle NaN.
    legacy_R5: >=2 hucre/dugum, agirliksiz, 0..89'da fit,
    >=5 fit dugumu, ic bosluklarda eski R5 interpolasyonu.
    """
    if method not in METHODS:
        raise ValueError(method)
    A = np.asarray(A, float).reshape(-1, 101)
    n = np.isfinite(A).sum(axis=0)
    raw = np.full(101, np.nan)
    if len(A):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            raw = np.nanmedian(A, axis=0)
    legacy = method == 'legacy_R5'
    fit = (n >= (2 if legacy else 1)) & (GRID < (90 if legacy else 101))
    pred = np.full(101, np.nan)
    status = 'ok'
    if len(A) < (2 if legacy else 1):
        status = 'insufficient_training_cells'
    elif fit.sum() < (5 if legacy else 2):
        status = 'insufficient_fit_nodes'
    else:
        model = IsotonicRegression(increasing=True, out_of_bounds='nan')
        model.fit(GRID[fit], raw[fit], sample_weight=None if legacy else n[fit])
        if legacy:
            pred[:90] = model.predict(GRID[:90])
        else:
            # predict(GRID) YOK: desteksiz dugumler doldurulmaz.
            pred[fit] = model.predict(GRID[fit])
    return {'pred': pred, 'support': n, 'raw': raw, 'status': status,
            'n_fit_nodes': int(fit.sum()), 'n_train_cells': len(A)}


def cell_arrays(d):
    return {cid: g.pivot(index='sample_int', columns='SoC', values='Voltage')
            .reindex(columns=GRID).sort_index() for cid, g in d.groupby('Condition_ID')}


def validate(d, analysis):
    groups = cell_arrays(d)
    empty = np.empty((0, 101))
    cache = {(m, cid): build_lut(groups[cid].to_numpy() if cid in groups else empty, m)
             for m in METHODS for cid in range(1, 25)}
    rows = []
    for cid, cells in groups.items():
        meta = META.set_index('Condition_ID').loc[cid]
        T, cutoff, rate = meta.Temperature, meta.Rate, meta.Charge_Rate
        experiments = [('LOCO', None, None, np.nan)]
        if T in (25, 45):
            low, high = ((10, 45) if T == 25 else (25, 60))
            experiments.append(('temperature_holdout', LOOKUP[low, cutoff, rate],
                                LOOKUP[high, cutoff, rate], (T-low)/(high-low)))
        if rate == 1.0:
            experiments.append(('rate_holdout', LOOKUP[T, cutoff, 0.7],
                                LOOKUP[T, cutoff, 2.0], (1.0-0.7)/(2.0-0.7)))
        for validation, lowcid, highcid, w in experiments:
            for sid, true in cells.iterrows():
                for method in METHODS:
                    if validation == 'LOCO':
                        a = build_lut(cells.drop(index=sid).to_numpy(), method)
                        b = None
                        pred = a['pred'].copy()
                    else:
                        a, b = cache[method, lowcid], cache[method, highcid]
                        # Bir taraf NaN ise sonuc NaN: tek tarafa tasima yok.
                        pred = (1-w)*a['pred'] + w*b['pred']
                    for soc in range(91):
                        y, p = float(true.loc[soc]), float(pred[soc])
                        if not np.isfinite(y):
                            reason = 'missing_test_target'
                        elif method == 'legacy_R5' and soc == 90:
                            reason = 'outside_legacy_domain'
                        elif np.isfinite(p):
                            reason = 'evaluable'
                        elif a['support'][soc] == 0 or (b is not None and b['support'][soc] == 0):
                            reason = 'zero_training_support'
                        elif a['status'] != 'ok' or (b is not None and b['status'] != 'ok'):
                            reason = 'insufficient_training_curve'
                        else:
                            reason = 'legacy_support_or_domain_limit'
                        rows.append(dict(analysis=analysis, validation=validation,
                            Condition_ID=cid, sample_int=int(sid), Temperature=T,
                            Rate=cutoff, Charge_Rate=rate, method=method, SoC=soc,
                            target_V=y, prediction_V=p, reason=reason,
                            train_condition_a=cid if lowcid is None else lowcid,
                            train_condition_b=highcid, weight_b=w,
                            support_a=int(a['support'][soc]),
                            support_b=int(b['support'][soc]) if b is not None else np.nan,
                            fit_status_a=a['status'], fit_status_b=b['status'] if b else '',
                            n_train_cells_a=a['n_train_cells'],
                            n_train_cells_b=b['n_train_cells'] if b else np.nan))
        print(f'{analysis}: kosul {cid}/24 tamamlandi.')
    return pd.DataFrame(rows)


def metrics(nodes, common=False):
    rows = []
    for key, g in nodes.groupby(KEYS + ['method'], sort=False):
        for band, lo, hi in [('all_0_90_inclusive', 0, 91),
                             ('[0,20)', 0, 20), ('[20,60)', 20, 60), ('[60,90]', 60, 91)]:
            q = g[g.SoC.between(lo, hi-1)]
            target = np.isfinite(q.target_V)
            paired = target & np.isfinite(q.prediction_V)
            if common:
                paired &= q.common_support
            n = int(paired.sum())
            eligible = n >= MIN_EVAL_POINTS
            err = (q.loc[paired, 'prediction_V'] - q.loc[paired, 'target_V']).to_numpy()*1000
            r = dict(zip(KEYS + ['method'], key))
            r.update(band=band, comparison='common_support' if common else 'own_support',
                Temperature=g.Temperature.iloc[0], Rate=g.Rate.iloc[0],
                Charge_Rate=g.Charge_Rate.iloc[0], n_grid=len(q), n_target=int(target.sum()),
                n_paired=n, n_used=n if eligible else 0, eligible=eligible,
                status='ok' if eligible else ('no_test_target' if not target.any() else
                       'no_prediction' if n == 0 else 'fewer_than_5_paired_nodes'),
                MAE_mV=float(np.mean(abs(err))) if eligible else np.nan,
                RMSE_mV=float(np.sqrt(np.mean(err**2))) if eligible else np.nan,
                bias_mV=float(np.mean(err)) if eligible else np.nan,
                max_abs_mV=float(np.max(abs(err))) if eligible else np.nan)
            rows.append(r)
    return pd.DataFrame(rows)


def summarize(m, by_condition=False):
    keys = ['analysis', 'validation', 'method', 'comparison', 'band']
    if by_condition:
        keys += ['Condition_ID', 'Temperature', 'Rate', 'Charge_Rate']
    s = m.groupby(keys, dropna=False).agg(
        n_planned_cells=('sample_int', 'size'), n_evaluated_cells=('eligible', 'sum'),
        n_grid=('n_grid', 'sum'), n_target=('n_target', 'sum'),
        n_paired=('n_paired', 'sum'), n_used=('n_used', 'sum'),
        MAE_mV=('MAE_mV', 'mean'), mean_cell_RMSE_mV=('RMSE_mV', 'mean'),
        bias_mV=('bias_mV', 'mean'), worst_abs_mV=('max_abs_mV', 'max')).reset_index()
    s['paired_target_fraction'] = s.n_paired / s.n_target.replace(0, np.nan)
    s['used_target_fraction'] = s.n_used / s.n_target.replace(0, np.nan)
    s['evaluated_cell_fraction'] = s.n_evaluated_cells / s.n_planned_cells
    return s


def export_luts(datasets, out):
    rows = []
    for analysis, d in datasets.items():
        groups = cell_arrays(d)
        for cid in range(1, 25):
            fit = build_lut(groups[cid].to_numpy() if cid in groups else np.empty((0, 101)))
            for soc in GRID:
                rows.append(dict(analysis=analysis, Condition_ID=cid, SoC=int(soc),
                    V_raw_median=fit['raw'][soc], V_lut=fit['pred'][soc],
                    n_cells_at_soc=int(fit['support'][soc]), fit_status=fit['status']))
    lut = pd.DataFrame(rows).merge(META, on='Condition_ID')
    lut.to_csv(out/'LUT_all_0_100.csv', index=False)
    lut[lut.SoC <= 90].to_csv(out/'LUT_all_0_90_inclusive.csv', index=False)
    final = lut[(lut.analysis == 'full_history') & (lut.SoC <= 90)]
    arr = final.pivot(index='Condition_ID', columns='SoC', values='V_lut').to_numpy()
    support = final.pivot(index='Condition_ID', columns='SoC', values='n_cells_at_soc').to_numpy()
    np.save(out/'final_LUT_float64.npy', arr)
    np.save(out/'final_LUT_float32.npy', arr.astype(np.float32))
    np.save(out/'final_LUT_float16.npy', arr.astype(np.float16))
    np.save(out/'final_LUT_valid_mask.npy', np.isfinite(arr))
    np.save(out/'final_LUT_support.npy', support.astype(np.int16))
    finite = np.isfinite(arr)
    qa = {'shape': list(arr.shape), 'finite_nodes': int(finite.sum()),
          'NaN_nodes': int((~finite).sum()), 'float32_array_bytes': arr.astype('float32').nbytes,
          'float16_array_bytes': arr.astype('float16').nbytes}
    for typ in ('float32', 'float16'):
        qa[typ+'_max_quantization_mV'] = (float(np.max(abs(arr[finite].astype(typ)
                 .astype(float)-arr[finite]))*1000) if finite.any() else None)
    (out/'LUT_QA.json').write_text(json.dumps(qa, indent=2), encoding='utf-8')
    return qa


def run(uploaded, out):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    missing = set(FILES.values()) - set(uploaded)
    if missing:
        raise ValueError(f'Eksik dosyalar: {sorted(missing)}')
    datasets = {a: load_profiles(uploaded[f], a) for a, f in FILES.items()}
    full_pairs = set(map(tuple, datasets['full_history'][['Condition_ID', 'sample_int']].values))
    broad_pairs = set(map(tuple, datasets['broad_52_132'][['Condition_ID', 'sample_int']].values))
    if not broad_pairs.issubset(full_pairs):
        raise ValueError('52-132 hucreleri tam gecmis dosyasinin alt kumesi olmali.')
    inventory = []
    for a, d in datasets.items():
        for cid in range(1, 25):
            g = d[d.Condition_ID == cid]
            inventory.append(dict(analysis=a, Condition_ID=cid, n_cells=g.sample_int.nunique(),
                                  n_finite_cell_nodes=int(np.isfinite(g.Voltage).sum())))
    pd.DataFrame(inventory).to_csv(out/'input_inventory.csv', index=False)
    nodes = pd.concat([validate(d, a) for a, d in datasets.items()], ignore_index=True)
    key = KEYS + ['SoC']
    common = nodes.assign(valid=np.isfinite(nodes.target_V) & np.isfinite(nodes.prediction_V))
    common = common.groupby(key).agg(common_support=('valid', 'all'), methods=('method', 'nunique'))
    assert common.methods.eq(2).all()
    nodes = nodes.merge(common[['common_support']], left_on=key, right_index=True, validate='many_to_one')
    nodes.to_csv(out/'validation_node_predictions.csv', index=False)
    m = pd.concat([metrics(nodes), metrics(nodes, common=True)], ignore_index=True)
    m.to_csv(out/'validation_cell_metrics.csv', index=False)
    summary = summarize(m)
    summary.to_csv(out/'validation_summary.csv', index=False)
    summarize(m, True).to_csv(out/'validation_summary_by_condition.csv', index=False)
    nodes.groupby(['analysis', 'validation', 'method', 'Condition_ID', 'reason']).size().rename(
        'n_nodes').reset_index().to_csv(out/'node_exclusion_reasons.csv', index=False)
    paired = m[m.comparison == 'common_support'].pivot(
        index=KEYS+['band'], columns='method', values=['MAE_mV', 'RMSE_mV', 'n_used'])
    paired.columns = ['_'.join(c) for c in paired.columns]
    paired['delta_MAE_unified_minus_legacy_mV'] = paired.MAE_mV_unified - paired.MAE_mV_legacy_R5
    paired.to_csv(out/'paired_method_comparison.csv')
    qa = export_luts(datasets, out)
    META.to_csv(out/'condition_map.csv', index=False)
    (out/'inputs').mkdir(exist_ok=True)
    for f in FILES.values():
        (out/'inputs'/f).write_bytes(uploaded[f])
    config = dict(timestamp_utc=datetime.now(timezone.utc).isoformat(),
        python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
        sklearn=sklearn.__version__, min_eval_points=MIN_EVAL_POINTS,
        fit_grid=[0, 100], output_grid=[0, 90], grid_step=1, min_support=1,
        minimum_supported_fit_nodes=2, zero_support='NaN; no filling',
        isotonic_weight='number of finite training cells',
        input_sha256={f: hashlib.sha256(uploaded[f]).hexdigest() for f in FILES.values()})
    (out/'run_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    (out/'README.txt').write_text(
        'Unified R5/R9 validation. Voltages in V; errors in mV.\n'
        'Inputs must be untrimmed 0..100 cell-median profiles from the existing audited pipeline.\n'
        'Rate = charge cutoff; Charge_Rate = discharge rate.\n'
        'The full-history LUT is the final artifact; the 52..132 LUT is a sensitivity artifact.\n'
        'All folds use training-only medians, support counts and weighted isotonic fits.\n'
        'Zero-support nodes remain NaN. Two supported fit nodes are required.\n'
        'Factor interpolation requires valid predictions from both bracketing conditions.\n'
        'A cell/band needs at least five paired nodes for a reported error.\n'
        'MAE and RMSE summaries are arithmetic means of evaluable cell-level metrics.\n'
        'Planned cells are those present in each input/window; absent conditions are in input_inventory.\n'
        'n_target: finite held-cell targets; n_paired: finite target/prediction pairs;\n'
        'n_used: paired nodes in cells/bands satisfying the five-node rule.\n'
        'own_support compares separate evaluation supports; common_support uses identical nodes.\n'
        'legacy_R5 reconstructs the reviewed R5 rules; it does not import archived results.\n'
        'Legacy excludes 90; unified includes 90. Thus common_support cannot include 90.\n'
        'Bands are [0,20), [20,60), [60,90]; the last includes 90 for unified.\n'
        'New masking also revises old R9 internal-gap interpolation.\n'
        'Validation uses float64 curves; float32/float16 conversion errors are reported separately.\n'
        'Final arrays: rows Condition_ID=1..24, columns SoC=0..90; NaNs must be respected.\n'
        'These are offline cycle-normalized SoC targets, not online SoC estimates.\n'
        'Upstream current-target selection/reconstruction is not rerun here; this is not\n'
        'evidence that all upstream preprocessing is training-only.\n', encoding='utf-8')
    print('\nYENI YONTEM / ANA SONUCLAR (mV):')
    print(summary[(summary.method == 'unified') & (summary.comparison == 'own_support') &
                  (summary.band == 'all_0_90_inclusive')].to_string(index=False))
    print('\nNihai LUT:', qa)
    return nodes, m, summary


def main():
    from google.colab import files
    print('Iki dosyayi birlikte secin:\n' + '\n'.join(FILES.values()))
    uploaded = files.upload()
    out = Path(tempfile.mkdtemp(prefix='SoC_unified_'))
    run(uploaded, out)
    archive = shutil.make_archive(str(out), 'zip', root_dir=out)
    print('\nTamamlandi. Sonuc ZIP dosyasi:', archive)
    files.download(archive)


# Original auto-run omitted: the runner below supplies audited local inputs.

# ---------- LOCAL, NONDESTRUCTIVE STAGE RUNNER ----------
source1=require_path(INPUT_CELLS)
source2=require_path(INPUT_BROAD)
out=stage_dir('03_unified')
print('SOURCE FULL:', source1)
print('SOURCE WINDOW:', source2)
run({source1.name:source1.read_bytes(), source2.name:source2.read_bytes()},out)
import pandas as pd
summary=pd.read_csv(out/'validation_summary.csv')
p=summary[(summary.method.eq('unified')) & (summary.comparison.eq('own_support')) & (summary.band.eq('all_0_90_inclusive'))]
print('UNIFIED 6 MAIN RESULTS:')
print(p.to_string(index=False))
stage_zip('03_unified','SoC_unified_')


## 04 — Fixed-prediction coverage and support-threshold sensitivity

Original `ConditionvCodes(2).py` lines **29,797–30,072**. Run after the prior required stage.

In [ ]:
# GOOGLE COLAB: Tamamini TEK hucreye yapistirin ve calistirin.
# Manuel girdi: onceki SoC_unified_....zip sonuc dosyasi.
# Sabit tahminler uzerinde kapsam denetimi; model YENIDEN EGITILMEZ.
import io, json, hashlib, tempfile, shutil, platform
from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

CELL_KEYS = ['analysis', 'validation', 'Condition_ID', 'sample_int']
BANDS = {'all_0_90_inclusive': (0, 90), '[0,20)': (0, 19),
         '[20,60)': (20, 59), '[60,90]': (60, 90)}
THRESHOLDS = (1, 2, 3)
MIN_POINTS = 5
SCOPES = ('all', 'target_C5', 'without_target_C5',
          'C5_involved', 'without_C5_involvement')


def read_csv(z, name):
    # Arsiv klasor icerse de calisir; belirsiz ayni adlari reddeder.
    hits = [p for p in z.namelist() if p == name or p.endswith('/'+name)]
    if len(hits) != 1:
        raise ValueError(f'ZIP icinde tek bir {name} bekleniyor; bulunan: {hits}')
    return pd.read_csv(z.open(hits[0]))


def load_nodes(z):
    d = read_csv(z, 'validation_node_predictions.csv')
    required = CELL_KEYS + ['method', 'SoC', 'target_V', 'prediction_V',
        'support_a', 'support_b', 'train_condition_a', 'train_condition_b',
        'Temperature', 'Rate', 'Charge_Rate']
    if not set(required).issubset(d):
        raise ValueError(f'Dugum dosyasinda eksik sutunlar: {set(required)-set(d)}')
    d = d[d.method.eq('unified')].copy()
    if d.empty or set(d.analysis) != {'full_history', 'broad_52_132'}:
        raise ValueError('Iki analiz kapsaminda unified sonuclari bulunamadi.')
    numeric = ['Condition_ID', 'sample_int', 'SoC', 'target_V', 'prediction_V',
               'support_a', 'support_b', 'train_condition_a', 'train_condition_b']
    for c in numeric:
        d[c] = pd.to_numeric(d[c], errors='raise')
    if d.duplicated(CELL_KEYS+['SoC']).any():
        raise ValueError('Yinelenen hucre-deney-SoC kaydi var.')
    if np.isinf(d[['target_V', 'prediction_V']].to_numpy()).any():
        raise ValueError('Sonsuz hedef/tahmin degeri var.')
    for key, g in d.groupby(CELL_KEYS):
        if set(g.SoC) != set(range(91)) or len(g) != 91:
            raise ValueError(f'{key}: 0..90 dugum kayitlari eksik.')
    factor = ~d.validation.eq('LOCO')
    if not set(d.validation).issubset({'LOCO', 'rate_holdout', 'temperature_holdout'}):
        raise ValueError('Bilinmeyen dogrulama turu.')
    for a in d.analysis.unique():
        if set(d.loc[d.analysis.eq(a), 'validation']) != {'LOCO','rate_holdout','temperature_holdout'}:
            raise ValueError(f'{a}: eksik dogrulama deneyi.')
    for col, mask in [('support_a', np.ones(len(d), dtype=bool)), ('support_b', factor)]:
        v=d.loc[mask,col].to_numpy(float)
        if not np.isfinite(v).all() or (v<0).any() or not np.allclose(v,np.round(v)):
            raise ValueError(f'{col}: gecersiz egitim destek sayisi.')
    d['effective_support'] = d.support_a
    d.loc[factor, 'effective_support'] = np.minimum(d.loc[factor,'support_a'], d.loc[factor,'support_b'])
    if (d.prediction_V.notna() & d.effective_support.eq(0)).any():
        raise ValueError('Sifir destekli dugumde tahmin var: ortak yontem kurali ihlal edilmis.')
    d['C5_involved'] = d.Condition_ID.eq(5) | d.train_condition_a.eq(5) | d.train_condition_b.eq(5)
    return d


def cell_metrics(d):
    rows=[]
    for key, g in d.groupby(CELL_KEYS, sort=True):
        for band, (lo, hi) in BANDS.items():
            q=g[g.SoC.between(lo,hi)]
            target=np.isfinite(q.target_V.to_numpy())
            pred=np.isfinite(q.prediction_V.to_numpy())
            paired=target & pred
            for threshold in THRESHOLDS:
                selected=paired & (q.effective_support.to_numpy() >= threshold)
                count=int(selected.sum())
                eligible=count >= MIN_POINTS
                err=(q.prediction_V.to_numpy()-q.target_V.to_numpy())[selected]*1000
                r=dict(zip(CELL_KEYS,key))
                r.update(band=band, support_threshold=threshold,
                    Temperature=g.Temperature.iloc[0], Rate=g.Rate.iloc[0],
                    Charge_Rate=g.Charge_Rate.iloc[0], C5_involved=bool(g.C5_involved.iloc[0]),
                    n_grid=len(q), n_target=int(target.sum()), n_missing_target=int((~target).sum()),
                    n_no_prediction=int((target & ~pred).sum()), n_paired=int(paired.sum()),
                    n_filtered_by_threshold=int((paired & ~selected).sum()), n_selected=count,
                    n_discarded_by_min_points=count if not eligible else 0,
                    n_used=count if eligible else 0, evaluated=eligible,
                    status='evaluated' if eligible else 'no_target' if not target.any() else
                           'no_prediction' if not paired.any() else
                           'no_nodes_at_threshold' if count==0 else 'fewer_than_5_nodes',
                    MAE_mV=float(np.mean(abs(err))) if eligible else np.nan,
                    RMSE_mV=float(np.sqrt(np.mean(err**2))) if eligible else np.nan,
                    bias_mV=float(np.mean(err)) if eligible else np.nan,
                    max_abs_mV=float(np.max(abs(err))) if eligible else np.nan)
                assert r['n_grid']==r['n_missing_target']+r['n_no_prediction']+r['n_filtered_by_threshold']+r['n_discarded_by_min_points']+r['n_used']
                rows.append(r)
    return pd.DataFrame(rows)


def scope_filter(m,scope):
    if scope=='all': return m
    if scope=='target_C5': return m[m.Condition_ID.eq(5)]
    if scope=='without_target_C5': return m[~m.Condition_ID.eq(5)]
    if scope=='C5_involved': return m[m.C5_involved]
    return m[~m.C5_involved]


def aggregate(m, by_condition=False, scopes=('all',)):
    rows=[]
    keys=['analysis','validation','band','support_threshold']
    if by_condition: keys+=['Condition_ID','Temperature','Rate','Charge_Rate']
    for scope in scopes:
        for key,g in scope_filter(m,scope).groupby(keys,sort=True):
            e=g[g.evaluated]
            r=dict(zip(keys,key));r['scope']=scope
            r.update(n_planned_cells=len(g),n_evaluated_cells=len(e),n_skipped_cells=len(g)-len(e))
            for c in ('n_grid','n_target','n_missing_target','n_no_prediction','n_paired',
                      'n_filtered_by_threshold','n_selected','n_discarded_by_min_points','n_used'):
                r[c]=int(g[c].sum())
            r.update(n_unscored_target=r['n_target']-r['n_used'],
                target_coverage_pct=100*r['n_used']/r['n_target'] if r['n_target'] else np.nan,
                grid_coverage_pct=100*r['n_used']/r['n_grid'],
                evaluated_cell_pct=100*len(e)/len(g),
                selected_nodes_min=int(g.n_selected.min()),
                selected_nodes_median=float(g.n_selected.median()),
                selected_nodes_max=int(g.n_selected.max()),
                evaluated_nodes_min=int(e.n_used.min()) if len(e) else np.nan,
                evaluated_nodes_median=float(e.n_used.median()) if len(e) else np.nan,
                evaluated_nodes_max=int(e.n_used.max()) if len(e) else np.nan,
                MAE_mV=e.MAE_mV.mean(), median_cell_MAE_mV=e.MAE_mV.median(),
                P90_cell_MAE_mV=e.MAE_mV.quantile(.9), mean_cell_RMSE_mV=e.RMSE_mV.mean(),
                bias_mV=e.bias_mV.mean(), worst_abs_mV=e.max_abs_mV.max())
            rows.append(r)
    return pd.DataFrame(rows)


def condition5_maps(z,d,out):
    allmaps=[]
    luts=read_csv(z,'LUT_all_0_100.csv')
    for analysis,filename in [('full_history','cell_profiles_nominal_concordant.csv'),
                              ('broad_52_132','broad_52_132_cell_profiles.csv')]:
        p=read_csv(z,'inputs/'+filename)
        vc='V_cell_median' if 'V_cell_median' in p else 'Voltage'
        p=p[p.Condition_ID.eq(5)]
        A=p.pivot(index='sample_int',columns='SoC',values=vc).reindex(columns=range(91)).sort_index()
        support=np.isfinite(A.to_numpy()).sum(axis=0)
        lut=luts[(luts.analysis==analysis)&(luts.Condition_ID==5)&(luts.SoC<=90)].sort_values('SoC')
        if len(lut)!=91 or not np.array_equal(support,lut.n_cells_at_soc.to_numpy()):
            raise ValueError(f'{analysis}: profil destegi ile LUT destegi uyusmuyor.')
        q=d[(d.analysis==analysis)&(d.validation=='LOCO')&(d.Condition_ID==5)].copy()
        q['all_cell_support']=q.SoC.map(dict(enumerate(support)))
        expected=q.all_cell_support - q.target_V.notna().astype(int)
        if not np.array_equal(expected.to_numpy(),q.support_a.to_numpy()):
            raise ValueError(f'{analysis}: LOCO egitim destegi beklenenden farkli.')
        q['evaluated_node']=q.target_V.notna() & q.prediction_V.notna()
        q['node_state']=np.where(q.target_V.isna(),0,np.where(q.prediction_V.isna(),1,2))
        allmaps.append(q)
        S=q.pivot(index='sample_int',columns='SoC',values='support_a').reindex(index=A.index,columns=range(91))
        C=q.pivot(index='sample_int',columns='SoC',values='node_state').reindex(index=A.index,columns=range(91))
        fig,axs=plt.subplots(3,1,figsize=(11,7.5),sharex=True,
                             gridspec_kw={'height_ratios':[1,1.5,1.5]},layout='constrained')
        axs[0].step(range(91),support,where='mid',color='#244e75')
        axs[0].set(ylabel='All-cell support',ylim=(-.3,len(A)+.5),
                   title=f'Condition 5: 10°C, C/5, 2C | {analysis}')
        im=axs[1].imshow(S,aspect='auto',interpolation='nearest',vmin=0,vmax=max(1,len(A)-1),cmap='viridis')
        fig.colorbar(im,ax=axs[1],label='Training cells',pad=.015)
        axs[1].set(ylabel='Held-out cell',title='Training support after excluding the held-out cell')
        cmap=ListedColormap(['#dadada','#d3624f','#3b8a78'])
        axs[2].imshow(C,aspect='auto',interpolation='nearest',cmap=cmap,norm=BoundaryNorm([-.5,.5,1.5,2.5],3))
        axs[2].set(ylabel='Held-out cell',xlabel='SoC (%)',title='Target availability and LOCO evaluation')
        for ax in axs[1:]:
            ax.set_yticks(range(len(A)),[str(int(x)) for x in A.index])
        axs[2].legend(handles=[Patch(color=c,label=l) for c,l in
            zip(cmap.colors,['Missing target','Target present; no prediction','Evaluable pair'])],
            loc='upper center',bbox_to_anchor=(.5,-.25),ncol=3,frameon=False)
        axs[2].set_xticks(range(0,91,10));axs[2].set_xlim(-.5,90.5)
        fig.savefig(out/f'Condition5_support_{analysis}.png',dpi=250)
        plt.close(fig)
    pd.concat(allmaps).to_csv(out/'Condition5_node_support_audit.csv',index=False)


def run_audit(zip_bytes,out):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    with ZipFile(io.BytesIO(zip_bytes)) as z:
        d=load_nodes(z)
        m=cell_metrics(d)
        s=aggregate(m,scopes=SCOPES)
        c=aggregate(m,by_condition=True)
        base=s[(s.scope=='all')&(s.support_threshold==1)]
        original=read_csv(z,'validation_summary.csv')
        original=original[(original.method=='unified')&(original.comparison=='own_support')]
        check=base.merge(original,on=['analysis','validation','band'],suffixes=('_audit','_original'),validate='one_to_one')
        if len(check)!=len(base) or len(check)!=len(original):
            raise ValueError('Referans ozet ile denetim gruplari uyusmuyor.')
        for field in ('n_planned_cells','n_evaluated_cells','n_grid','n_target','n_paired','n_used','MAE_mV','bias_mV'):
            if not np.allclose(check[field+'_audit'],check[field+'_original'],rtol=0,atol=1e-8,equal_nan=True):
                raise ValueError(f'Referans sonucla uyusmazlik: {field}')
        m.to_csv(out/'cell_coverage_and_threshold_metrics.csv',index=False)
        base.to_csv(out/'primary_coverage_all_bands.csv',index=False)
        base[base.band=='all_0_90_inclusive'].to_csv(out/'primary_coverage_overall.csv',index=False)
        c.to_csv(out/'coverage_by_condition_and_threshold.csv',index=False)
        s.to_csv(out/'support_threshold_and_C5_sensitivity.csv',index=False)
        m.groupby(['analysis','validation','band','support_threshold','status']).size().rename('n_cells').reset_index().to_csv(out/'cell_exclusion_reasons.csv',index=False)
        d.to_csv(out/'audited_unified_nodes.csv',index=False)
        condition5_maps(z,d,out)
        # Pencereye girmeyen hucreleri, dogrulamada atlanan hucrelerden ayir.
        fp=read_csv(z,'inputs/cell_profiles_nominal_concordant.csv')
        bp=read_csv(z,'inputs/broad_52_132_cell_profiles.csv')
        pairs=['Condition_ID','sample_int']
        inventory=fp[pairs].drop_duplicates().merge(bp[pairs].drop_duplicates().assign(present_in_broad=True),on=pairs,how='left')
        inventory['present_in_broad']=inventory.present_in_broad.eq(True)
        inventory.to_csv(out/'cell_window_inventory.csv',index=False)
    config={'source_zip_sha256':hashlib.sha256(zip_bytes).hexdigest(),
        'timestamp_utc':datetime.now(timezone.utc).isoformat(),'python':platform.python_version(),
        'pandas':pd.__version__,'numpy':np.__version__,'support_thresholds':list(THRESHOLDS),
        'minimum_paired_nodes':MIN_POINTS,'refitted':False,'reference_summary_check':'passed'}
    (out/'audit_config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    (out/'README.txt').write_text(
        'KAPSAM DENETIMI: mevcut unified tahminleri sabittir; yeniden egitim yoktur.\n'
        'Her hata degeri hucre bazinda hesaplanir; ozet MAE hucrelere esit agirlik verir.\n'
        'Egitim destegi LOCO icin support_a, faktor tutmada min(support_a,support_b).\n'
        'Esik 1 ana yontem; 2 ve 3 yalnizca degerlendirme kapsam duyarliligidir.\n'
        'Esik altindaki dugumler fitten silinmez; bu bir yeniden egitim/alternatif LUT deneyi degildir.\n'
        'Her hucre/bant/esik icin en az 5 nokta gerekir. Daha azinda MAE NaN, asla sifir degildir.\n'
        'n_grid = n_missing_target + n_no_prediction + n_filtered_by_threshold +\n'
        'n_discarded_by_min_points + n_used.\n'
        'target_coverage_pct paydasi n_target; grid_coverage_pct paydasi n_grid.\n'
        'Planlanan hucreler ilgili giriste bulunanlardir. Pencereye girmeyenler inventory dosyasindadir.\n'
        'n_paired esik oncesi; n_selected esik sonrasi; n_used bes-nokta kurali sonrasi sayidir.\n'
        'selected_nodes_* tum planlanan hucreler; evaluated_nodes_* yalnizca puanlanan hucreler.\n'
        'all: tum hedef kosullar; target_C5: yalniz C5; without_target_C5: C5 hedefi haric.\n'
        'C5_involved: C5 hedef veya egitim komsusu; without_C5_involvement: bunlar haric.\n'
        'C5 hedefi faktor tutmada yoktur; ilgili bos scope gruplari satir olarak uretilmez.\n'
        'C5 haric ortalama farki farkli hucre karisimidir; iyilesme veya nedensel etki degildir.\n'
        'Esikler degistikce dugumler ve bazi hucreler degisebilir; hatayi kapsamla birlikte okuyun.\n'
        'Condition5 haritalari girdi profillerinden R11 tipi destek sayisini yeniden uretir;\n'
        'arsivlenmis eski R11 dosyasi okunmamistir. Destek sayilari LUT ve LOCO kayitlariyla dogrulanir.\n'
        'Harita yesil: sonlu hedef/tahmin cifti. Puanlanabilirlik cell metric tablosunda ayrica bulunur.\n'
        'Hata birimi mV. Toplam 0..90 dahil; son hata bandi [60,90] dahil.\n'
        'Bu hata bantlari cikarimsal [60,90) bant kapsamiyla karistirilmamalidir.\n'
        'Tahmin uretilmeyen dugumlerdeki hata bu analizden hesaplanamaz.\n',encoding='utf-8')
    main=base[base.band=='all_0_90_inclusive']
    cols=['analysis','validation','n_planned_cells','n_evaluated_cells','n_skipped_cells',
          'n_grid','n_missing_target','n_target','n_no_prediction','n_used','target_coverage_pct',
          'evaluated_nodes_min','evaluated_nodes_median','evaluated_nodes_max','MAE_mV']
    print('\nANA KAPSAM TABLOSU\n',main[cols].to_string(index=False))
    q=s[(s.scope=='all')&(s.band=='all_0_90_inclusive')]
    print('\nSABIT TAHMINLER / DESTEK ESIGI DUYARLILIGI\n',q[['analysis','validation','support_threshold','n_evaluated_cells','n_used','target_coverage_pct','MAE_mV']].to_string(index=False))
    print('\nKontroller gecti: referans hatalar, dugum sayimlari, C5 LUT ve LOCO destekleri.')
    return m,s,c


def main():
    from google.colab import files
    print('Onceki SoC_unified_....zip sonuc dosyasini secin (tek ZIP).')
    uploaded=files.upload()
    if len(uploaded)!=1:
        raise ValueError('Yalnizca bir sonuc ZIP dosyasi yukleyin.')
    name,blob=next(iter(uploaded.items()))
    # Colab adina (1), (2) eklese de dogrudan yuklenen baytlar kullanilir.
    if not name.lower().endswith('.zip'):
        raise ValueError('Girdi bir ZIP dosyasi olmali.')
    out=Path(tempfile.mkdtemp(prefix='SoC_coverage_audit_'))
    run_audit(blob,out)
    archive=shutil.make_archive(str(out),'zip',root_dir=out)
    print('\nTamamlandi:',archive)
    files.download(archive)


# Original auto-run omitted: the runner below supplies audited local inputs.

# ---------- LOCAL, NONDESTRUCTIVE STAGE RUNNER ----------
path=require_path('/content/SoC_unified_ConditionvCodes2.zip')
out=stage_dir('04_unified_coverage_audit')
run_audit(path.read_bytes(),out)
print('AUDIT OUTPUTS:',[p.name for p in sorted(out.iterdir())])
stage_zip('04_unified_coverage_audit','SoC_coverage_audit_')


## 05 — Five-representation common-node comparison

Original `ConditionvCodes(2).py` lines **30,702–32,549**. Run after the prior required stage.

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
# GOOGLE COLAB — TAMAMINI TEK HÜCREYE YAPIŞTIRIP ÇALIŞTIRIN.
#
# Zorunlu manuel girdi:
#   SoC_unified_xaevhp2i.zip
#
# İsteğe bağlı olarak aynı anda şu CSV'leri de seçebilirsiniz:
#   cell_profiles_nominal_concordant.csv
#   broad_52_132_cell_profiles.csv
#
# CSV seçilmezse ZIP içindeki inputs/ klasöründen okunur.
# Colab'ın dosya adına eklediği " (1)", " (2)" ekleri kabul edilir.
#
# Karşılaştırmalar:
#   1. global_iso       : tek ortak izotonik eğri
#   2. temperature_iso  : yalnız sıcaklığa bağlı izotonik eğriler
#   3. rate_iso         : yalnız deşarj hızına bağlı izotonik eğriler
#   4. condition_raw    : koşula özgü hücre medyanı, izotonik uygulanmaz
#   5. condition_iso    : mevcut birleşik izotonik LUT
#
# Doğrulama:
#   LOCO, hız tutma, sıcaklık tutma
#   Tam geçmiş ve 52–132 ortak pencere
#
# Ana karşılaştırma:
#   Beş yöntemin ortak hücre–SoC düğümleri
#   Hücre başına en az 5 değerlendirilebilir düğüm
#   Hücre MAE'lerinin aritmetik ortalaması
#
# Bootstrap:
#   Sabit test tahminlerindeki eşleştirilmiş hata farklarının
#   koşul-tabakalı fiziksel hücre bootstrap'ı; 2000 tekrar.
#   Modeller bootstrap içinde yeniden eğitilmez.

import io
import re
import json
import hashlib
import warnings
import shutil
import tempfile
import platform
import zipfile

from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations

import numpy as np
import pandas as pd
import sklearn
from sklearn.isotonic import IsotonicRegression


# ============================================================
# AYARLAR
# ============================================================

BOOTSTRAP_REPS = 2000
SEED = 20260928
MIN_EVAL_POINTS = 5
NUMERIC_TIE_MV = 1e-9

GRID = np.arange(101)

MODELS = (
    "global_iso",
    "temperature_iso",
    "rate_iso",
    "condition_raw",
    "condition_iso",
)

FILES = {
    "full_history": "cell_profiles_nominal_concordant.csv",
    "broad_52_132": "broad_52_132_cell_profiles.csv",
}

CELL_KEY = [
    "analysis",
    "validation",
    "Condition_ID",
    "sample_int",
]

NODE_KEY = CELL_KEY + ["SoC"]

# Bunlar LUT doğrulamasının yardımcı hata bantlarıdır.
# Son bant 90'ı içerir; kontrast analizindeki [60,90) ile karıştırmayın.
BANDS = [
    ("all_0_90_inclusive", 0, 91),
    ("[0,20)", 0, 20),
    ("[20,60)", 20, 60),
    ("[60,90]", 60, 91),
]

META = pd.DataFrame(
    [
        (1 + 6*t + 2*r + k, T, cutoff, rate)
        for t, T in enumerate((10, 25, 45, 60))
        for r, rate in enumerate((0.7, 1.0, 2.0))
        for k, cutoff in enumerate(("C/5", "C/40"))
    ],
    columns=[
        "Condition_ID",
        "Temperature",
        "Rate",
        "Charge_Rate",
    ],
)

MAP = META.set_index("Condition_ID")

LOOKUP = {
    (r.Temperature, r.Rate, r.Charge_Rate): r.Condition_ID
    for r in META.itertuples()
}


# ============================================================
# YARDIMCILAR VE GİRDİ KONTROLLERİ
# ============================================================

def require(ok, message):
    if not bool(ok):
        raise ValueError(message)


def csv_save(df, out, name):
    df.to_csv(Path(out) / name, index=False)


def canonical(name):
    p = Path(name)
    stem = re.sub(r"\s*\(\d+\)$", "", p.stem).lower()
    return stem + p.suffix.lower()


def member(z, basename):
    found = [
        n for n in z.namelist()
        if not n.endswith("/") and Path(n).name == basename
    ]

    require(
        len(found) == 1,
        f"ZIP: {basename} tekil bulunamadı: {found}",
    )

    info = z.getinfo(found[0])

    require(
        info.file_size < 300_000_000,
        f"ZIP üyesi beklenenden büyük: {basename}",
    )

    return z.read(found[0])


def load_profiles(blob, name):
    d = pd.read_csv(io.BytesIO(blob))
    d.columns = d.columns.str.strip()

    if "V_cell_median" in d:
        require(
            "Voltage" not in d,
            f"{name}: iki voltaj sütunu var; belirsiz girdi.",
        )
        d = d.rename(columns={"V_cell_median": "Voltage"})

    required = ["Condition_ID", "sample_int", "SoC", "Voltage"]

    require(
        set(required).issubset(d.columns),
        f"{name}: gerekli sütunlar: {required}",
    )

    for c in required:
        d[c] = pd.to_numeric(d[c], errors="raise")

    for c in required[:3]:
        a = d[c].to_numpy(float)

        require(
            np.isfinite(a).all()
            and np.allclose(a, np.round(a), rtol=0, atol=1e-8),
            f"{name}: {c} tam sayı ve sonlu olmalı.",
        )

        d[c] = np.round(a).astype(int)

    require(
        not np.isinf(d.Voltage).any(),
        f"{name}: sonsuz voltaj değeri var.",
    )

    require(
        d.Condition_ID.between(1, 24).all()
        and d.SoC.between(0, 100).all(),
        f"{name}: koşul veya SoC aralığı hatalı.",
    )

    require(
        not d.duplicated(required[:3]).any(),
        f"{name}: yinelenen hücre–koşul–SoC kaydı var.",
    )

    require(
        not (d.groupby("sample_int").Condition_ID.nunique() > 1).any(),
        f"{name}: bir fiziksel hücre birden çok koşulda.",
    )

    for (cid, sid), g in d.groupby(["Condition_ID", "sample_int"]):
        if cid <= 22:
            allowed = range(8*(cid-1)+1, 8*cid+1)
        elif cid == 23:
            allowed = range(177, 180)
        else:
            allowed = range(185, 188)

        require(
            sid in allowed,
            f"{name}: nominal harita uyuşmazlığı: koşul={cid}, hücre={sid}",
        )

        require(
            set(g.SoC) == set(GRID),
            f"{name}: hücre {sid} için 0..100 satırları eksik. "
            "Kırpılmamış profil dosyasını kullanın; voltajlar NaN olabilir.",
        )

    if name == "full_history":
        require(
            set(d.Condition_ID) == set(range(1, 25)),
            "Tam geçmiş dosyasında 24 koşulun tamamı bulunmalı.",
        )

    # Kaynak değişken eşlemesi:
    # Rate        = şarj sonlandırma akımı
    # Charge_Rate = deşarj C-oranı
    for c in ("Temperature", "Rate", "Charge_Rate"):
        if c not in d:
            continue

        expected = d.Condition_ID.map(MAP[c])

        if c == "Rate":
            actual = (
                d[c].astype(str)
                .str.replace(" ", "", regex=False)
                .str.upper()
            )
            ok = actual.eq(expected)
        else:
            actual = d[c].astype(str)
            actual = actual.str.replace("°", "", regex=False)
            actual = actual.str.replace("º", "", regex=False)
            actual = actual.str.replace("C", "", regex=False)
            actual = pd.to_numeric(actual.str.strip(), errors="coerce")

            ok = np.isclose(
                actual,
                expected,
                rtol=0,
                atol=1e-8,
            )

        require(
            np.asarray(ok).all(),
            f"{name}: {c} metadata uyuşmazlığı.",
        )

    return d[required].merge(
        META,
        on="Condition_ID",
        validate="many_to_one",
    )


def load_inputs(uploaded):
    zipnames = [
        n for n in uploaded
        if n.lower().endswith(".zip")
    ]

    require(
        len(zipnames) == 1,
        "Yalnız bir önceki SoC_unified_....zip dosyasını seçin.",
    )

    z = zipfile.ZipFile(io.BytesIO(uploaded[zipnames[0]]))

    old_config = json.loads(member(z, "run_config.json"))

    datasets = {}
    hashes = {}
    provenance = {}

    for analysis, filename in FILES.items():
        matched = [
            n for n in uploaded
            if canonical(n) == filename.lower()
        ]

        require(
            len(matched) <= 1,
            f"Aynı CSV birden fazla seçilmiş: {matched}",
        )

        if matched:
            blob = uploaded[matched[0]]
            provenance[analysis] = matched[0]
        else:
            blob = member(z, filename)
            provenance[analysis] = "reference_zip/inputs/" + filename

        hashes[filename] = hashlib.sha256(blob).hexdigest()
        expected = old_config.get("input_sha256", {}).get(filename)

        require(
            expected is not None,
            f"Referans ZIP girdi hash kaydı eksik: {filename}",
        )

        require(
            hashes[filename] == expected,
            f"{filename}: önceki çalışmanın girdisiyle SHA256 uyuşmuyor. "
            "Aynı dosyayı kullanın.",
        )

        datasets[analysis] = load_profiles(blob, analysis)

    full = datasets["full_history"][
        ["Condition_ID", "sample_int"]
    ].drop_duplicates()

    broad = datasets["broad_52_132"][
        ["Condition_ID", "sample_int"]
    ].drop_duplicates()

    require(
        set(map(tuple, broad.values)) <= set(map(tuple, full.values)),
        "Ortak pencere hücreleri tam geçmişin alt kümesi olmalı.",
    )

    reference = pd.read_csv(
        io.BytesIO(member(z, "validation_node_predictions.csv"))
    )
    reference = reference[reference.method.eq("unified")].copy()

    summary = pd.read_csv(
        io.BytesIO(member(z, "validation_summary.csv"))
    )
    summary = summary[
        summary.method.eq("unified")
        & summary.comparison.eq("own_support")
    ].copy()

    lut = np.load(
        io.BytesIO(member(z, "final_LUT_float64.npy")),
        allow_pickle=False,
    )

    support = np.load(
        io.BytesIO(member(z, "final_LUT_support.npy")),
        allow_pickle=False,
    )

    refhash = hashlib.sha256(
        uploaded[zipnames[0]]
    ).hexdigest()

    z.close()

    return (
        datasets,
        reference,
        summary,
        lut,
        support,
        hashes,
        provenance,
        refhash,
    )


# ============================================================
# ORTAK EĞRİ KURMA FONKSİYONU
# ============================================================

def build_curve(A, raw=False):
    """
    A: benzersiz eğitim hücreleri x 101 SoC düğümü.

    Bir satır bir fiziksel hücrenin medyan profilidir.
    Fit alanı: 0..100.
    Minimum destek: düğüm başına 1 eğitim hücresi.
    Minimum destekli fit düğümü: 2.
    Sıfır destekli düğüm: NaN, doldurma yok.

    raw=False:
        Hücre sayısıyla ağırlıklı izotonik regresyon.

    raw=True:
        Aynı eğitim hücrelerinin düğüm medyanı.
        Destek ve minimum fit düğümü şartları aynıdır.
    """
    A = np.asarray(A, dtype=float).reshape(-1, 101)

    n = np.isfinite(A).sum(axis=0)
    med = np.full(101, np.nan)

    if len(A):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            med = np.nanmedian(A, axis=0)

    ok = n >= 1
    p = np.full(101, np.nan)
    status = "ok"

    if not len(A):
        status = "insufficient_training_cells"
    elif ok.sum() < 2:
        status = "insufficient_fit_nodes"
    elif raw:
        p[ok] = med[ok]
    else:
        iso = IsotonicRegression(
            increasing=True,
            out_of_bounds="nan",
        )

        iso.fit(
            GRID[ok],
            med[ok],
            sample_weight=n[ok],
        )

        p[ok] = iso.predict(GRID[ok])

        require(
            np.all(np.diff(p[ok]) >= -1e-12),
            "Monotonluk kontrolü başarısız.",
        )

    require(
        not np.isfinite(p[n == 0]).any(),
        "Sıfır destekli düğüm doldurulmuş.",
    )

    return {
        "pred": p,
        "support": n,
        "raw": med,
        "status": status,
        "n_train_cells": len(A),
        "n_fit_nodes": int(ok.sum()),
    }


class Fold:
    """
    Tutulan hücre veya faktör düzeyi baştan çıkarılır.
    Beş temsil de bu eğitim kümesinden kurulur.
    """

    def __init__(self, frame, analysis, fold_id, audit):
        self.frame = frame
        self.analysis = analysis
        self.fold_id = fold_id
        self.audit = audit
        self.cache = {}

    def curve(self, model, cid, T, rate):
        if model == "global_iso":
            key = (model, "all")
            mask = np.ones(len(self.frame), bool)

        elif model == "temperature_iso":
            key = (model, float(T))
            mask = self.frame.Temperature.eq(T)

        elif model == "rate_iso":
            key = (model, float(rate))
            mask = self.frame.Charge_Rate.eq(rate)

        else:
            key = (model, int(cid))
            mask = self.frame.Condition_ID.eq(cid)

        if key not in self.cache:
            g = self.frame.loc[mask]
            A = g[list(GRID)].to_numpy(float)

            fit = build_curve(
                A,
                raw=(model == "condition_raw"),
            )

            fit["fit_id"] = len(self.audit)

            self.audit.append({
                "fit_id": fit["fit_id"],
                "analysis": self.analysis,
                "fold": self.fold_id,
                "model": model,
                "group": str(key[1]),
                "n_train_cells": fit["n_train_cells"],
                "n_fit_nodes": fit["n_fit_nodes"],
                "status": fit["status"],
                "training_cells": ";".join(
                    map(str, g.index.tolist())
                ),
                "training_conditions": ";".join(
                    map(str, sorted(g.Condition_ID.unique()))
                ),
            })

            self.cache[key] = fit

        return self.cache[key]

    def predict(self, model, cid, validation):
        row = MAP.loc[cid]

        T = row.Temperature
        rate = row.Charge_Rate
        cutoff = row.Rate

        uses_temperature = model in (
            "temperature_iso",
            "condition_raw",
            "condition_iso",
        )

        uses_rate = model in (
            "rate_iso",
            "condition_raw",
            "condition_iso",
        )

        if validation == "temperature_holdout" and uses_temperature:
            lo, hi = (10, 45) if T == 25 else (25, 60)
            w = (T - lo) / (hi - lo)

            a = self.curve(
                model,
                LOOKUP[lo, cutoff, rate],
                lo,
                rate,
            )

            b = self.curve(
                model,
                LOOKUP[hi, cutoff, rate],
                hi,
                rate,
            )

        elif validation == "rate_holdout" and uses_rate:
            lo, hi = 0.7, 2.0
            w = (1.0 - lo) / (hi - lo)

            a = self.curve(
                model,
                LOOKUP[T, cutoff, lo],
                T,
                lo,
            )

            b = self.curve(
                model,
                LOOKUP[T, cutoff, hi],
                T,
                hi,
            )

        else:
            w = np.nan
            a = self.curve(model, cid, T, rate)
            b = None

        if b is None:
            p = a["pred"].copy()
        else:
            # Bir komşu NaN ise sonuç NaN.
            # Tek taraftan taşıma yapılmaz.
            p = (1 - w)*a["pred"] + w*b["pred"]

        return p, a, b, w


# ============================================================
# TAHMİNLER
# ============================================================

def predict_dataset(d, analysis):
    frame = d.pivot(
        index="sample_int",
        columns="SoC",
        values="Voltage",
    ).reindex(columns=GRID)

    metadata = (
        d.drop_duplicates("sample_int")
        .set_index("sample_int")[
            ["Condition_ID", "Temperature", "Rate", "Charge_Rate"]
        ]
    )

    frame = frame.join(metadata).sort_index()

    rows = []
    audit = []
    factor_folds = {}

    specifications = [
        ("rate_holdout", "Charge_Rate", [1.0]),
        ("temperature_holdout", "Temperature", [25, 45]),
    ]

    for validation, column, levels in specifications:
        for level in levels:
            train = frame.loc[frame[column].ne(level)].copy()

            require(
                not train[column].eq(level).any(),
                "Tutulan faktör eğitimde kaldı.",
            )

            factor_folds[validation, level] = Fold(
                train,
                analysis,
                f"{validation}_{level}",
                audit,
            )

    for k, (sid, target) in enumerate(frame.iterrows(), 1):
        cid = int(target.Condition_ID)

        loco = Fold(
            frame.drop(index=sid),
            analysis,
            f"LOCO_{sid}",
            audit,
        )

        require(
            sid not in loco.frame.index,
            "LOCO test hücresi eğitimde kaldı.",
        )

        tasks = [("LOCO", loco)]

        if target.Charge_Rate == 1.0:
            tasks.append((
                "rate_holdout",
                factor_folds["rate_holdout", 1.0],
            ))

        if target.Temperature in (25, 45):
            tasks.append((
                "temperature_holdout",
                factor_folds[
                    "temperature_holdout",
                    target.Temperature,
                ],
            ))

        for validation, fold in tasks:
            q = pd.DataFrame({
                "analysis": analysis,
                "validation": validation,
                "Condition_ID": cid,
                "sample_int": int(sid),
                "SoC": np.arange(91),
                "Temperature": target.Temperature,
                "Rate": target.Rate,
                "Charge_Rate": target.Charge_Rate,
                "target_V": target[
                    list(range(91))
                ].to_numpy(float),
            })

            for model in MODELS:
                p, a, b, w = fold.predict(
                    model,
                    cid,
                    validation,
                )

                q[model] = p[:91]
                q[model + "_support_a"] = a["support"][:91]
                q[model + "_support_b"] = (
                    b["support"][:91]
                    if b is not None else np.nan
                )
                q[model + "_fit_a"] = a["fit_id"]
                q[model + "_fit_b"] = (
                    b["fit_id"]
                    if b is not None else np.nan
                )
                q[model + "_weight_b"] = w

                reason = np.full(
                    91,
                    "evaluable",
                    dtype=object,
                )

                reason[~np.isfinite(p[:91])] = (
                    "insufficient_training_curve"
                )

                zero = a["support"][:91] == 0

                if b is not None:
                    zero |= b["support"][:91] == 0

                reason[zero] = "zero_training_support"

                reason[
                    ~np.isfinite(q.target_V.to_numpy())
                ] = "missing_test_target"

                q[model + "_reason"] = reason

            q["common_five"] = (
                np.isfinite(q.target_V)
                & np.isfinite(q[list(MODELS)]).all(axis=1)
            )

            require(
                np.array_equal(
                    np.isfinite(q.condition_raw),
                    np.isfinite(q.condition_iso),
                ),
                "Ham/izotonik destek maskeleri farklı.",
            )

            rows.append(q)

        if k % 20 == 0 or k == len(frame):
            print(
                f"{analysis}: {k}/{len(frame)} hücre tamamlandı.",
                flush=True,
            )

    return (
        pd.concat(rows, ignore_index=True),
        pd.DataFrame(audit),
    )


# ============================================================
# ESKİ SONUÇLARI YENİDEN ÜRETME KONTROLLERİ
# ============================================================

def check_reference(nodes, reference, out):
    require(
        not reference.duplicated(NODE_KEY).any(),
        "Referans düğüm anahtarı yineleniyor.",
    )

    require(
        not nodes.duplicated(NODE_KEY).any(),
        "Yeni düğüm anahtarı yineleniyor.",
    )

    old = reference[
        NODE_KEY + ["target_V", "prediction_V"]
    ]

    new = nodes[
        NODE_KEY + ["target_V", "condition_iso"]
    ]

    q = new.merge(
        old,
        on=NODE_KEY,
        how="outer",
        suffixes=("_new", "_ref"),
        indicator=True,
        validate="one_to_one",
    )

    require(
        q["_merge"].eq("both").all(),
        "Referansla test hücresi/düğümü kümesi farklı.",
    )

    checks = []

    quantities = [
        ("target", "target_V_new", "target_V_ref"),
        ("prediction", "condition_iso", "prediction_V"),
    ]

    for kind, x, y in quantities:
        for key, g in q.groupby(["analysis", "validation"]):
            v = g[x].to_numpy(float)
            u = g[y].to_numpy(float)

            same = np.array_equal(
                np.isfinite(v),
                np.isfinite(u),
            )

            mask = np.isfinite(v) & np.isfinite(u)

            error = (
                float(np.max(np.abs(v[mask] - u[mask]))*1000)
                if mask.any() else 0.0
            )

            checks.append({
                "analysis": key[0],
                "validation": key[1],
                "quantity": kind,
                "same_finite_mask": same,
                "max_abs_difference_mV": error,
                "n_nodes": len(g),
            })

    checks = pd.DataFrame(checks)
    csv_save(checks, out, "reference_node_check.csv")

    require(
        checks.same_finite_mask.all()
        and checks.max_abs_difference_mV.le(1e-6).all(),
        "Referans tahminleri/gerçekleri yeniden üretilemedi.",
    )

    print(
        "Referans düğüm kontrolü geçti; en büyük fark (mV):",
        checks.max_abs_difference_mV.max(),
    )


# ============================================================
# HÜCRE HATALARI VE KAPSAM
# ============================================================

def metric_row(q, prediction, mask):
    n = int(mask.sum())
    eligible = n >= MIN_EVAL_POINTS

    errors = (
        prediction[mask]
        - q.target_V.to_numpy(float)[mask]
    )*1000

    n_target = int(np.isfinite(q.target_V).sum())

    return {
        "n_grid": len(q),
        "n_target": n_target,
        "n_paired": n,
        "n_used": n if eligible else 0,
        "eligible": eligible,
        "n_missing_target": len(q) - n_target,
        "n_target_not_paired": n_target - n,
        "n_pairs_discarded_min_nodes": n if not eligible else 0,
        "MAE_mV": (
            float(np.mean(abs(errors)))
            if eligible else np.nan
        ),
        "RMSE_mV": (
            float(np.sqrt(np.mean(errors**2)))
            if eligible else np.nan
        ),
        "bias_mV": (
            float(np.mean(errors))
            if eligible else np.nan
        ),
        "max_abs_mV": (
            float(np.max(abs(errors)))
            if eligible else np.nan
        ),
    }


def compute_metrics(nodes):
    rows = []
    pairs = []

    for key, g in nodes.groupby(CELL_KEY, sort=True):
        base = dict(zip(CELL_KEY, key))

        base.update(
            Temperature=g.Temperature.iloc[0],
            Rate=g.Rate.iloc[0],
            Charge_Rate=g.Charge_Rate.iloc[0],
        )

        for band, lo, hi in BANDS:
            q = g[g.SoC.between(lo, hi-1)]

            target = np.isfinite(
                q.target_V.to_numpy(float)
            )
            common = q.common_five.to_numpy(bool)

            for model in MODELS:
                p = q[model].to_numpy(float)
                own = target & np.isfinite(p)

                for scope, mask in [
                    ("own_support", own),
                    ("common_five", common),
                ]:
                    rows.append({
                        **base,
                        "band": band,
                        "model": model,
                        "comparison": scope,
                        **metric_row(q, p, mask),
                    })

            # İkili karşılaştırmalar tam 0..90 alanında yapılır.
            if band != "all_0_90_inclusive":
                continue

            for left, right in combinations(MODELS, 2):
                p = q[left].to_numpy(float)
                r = q[right].to_numpy(float)

                pairwise = (
                    target
                    & np.isfinite(p)
                    & np.isfinite(r)
                )

                for scope, mask in [
                    ("common_five", common),
                    ("pairwise_common", pairwise),
                ]:
                    L = metric_row(q, p, mask)
                    R = metric_row(q, r, mask)

                    pairs.append({
                        **base,
                        "band": band,
                        "comparison": scope,
                        "left_model": left,
                        "right_model": right,
                        "n_grid": len(q),
                        "n_target": int(target.sum()),
                        "n_paired": L["n_paired"],
                        "n_used": L["n_used"],
                        "eligible": L["eligible"],
                        "left_MAE_mV": L["MAE_mV"],
                        "right_MAE_mV": R["MAE_mV"],
                        # Pozitif: sağdaki yöntem daha iyi.
                        "gain_right_mV": (
                            L["MAE_mV"] - R["MAE_mV"]
                        ),
                    })

    return pd.DataFrame(rows), pd.DataFrame(pairs)


def aggregate(metrics, condition=False):
    keys = [
        "analysis",
        "validation",
        "comparison",
        "band",
        "model",
    ]

    if condition:
        keys += [
            "Condition_ID",
            "Temperature",
            "Rate",
            "Charge_Rate",
        ]

    s = metrics.groupby(keys, dropna=False).agg(
        n_planned_cells=("sample_int", "size"),
        n_evaluated_cells=("eligible", "sum"),
        n_grid=("n_grid", "sum"),
        n_target=("n_target", "sum"),
        n_paired=("n_paired", "sum"),
        n_used=("n_used", "sum"),
        n_missing_target=("n_missing_target", "sum"),
        n_target_not_paired=("n_target_not_paired", "sum"),
        n_pairs_discarded_min_nodes=(
            "n_pairs_discarded_min_nodes", "sum"
        ),
        MAE_mV=("MAE_mV", "mean"),
        mean_cell_RMSE_mV=("RMSE_mV", "mean"),
        bias_mV=("bias_mV", "mean"),
        worst_abs_mV=("max_abs_mV", "max"),
    ).reset_index()

    s["n_skipped_cells"] = (
        s.n_planned_cells - s.n_evaluated_cells
    )
    s["n_grid_not_used"] = s.n_grid - s.n_used

    s["target_coverage_pct"] = (
        100*s.n_used / s.n_target.replace(0, np.nan)
    )
    s["nominal_grid_coverage_pct"] = (
        100*s.n_used / s.n_grid.replace(0, np.nan)
    )
    s["cell_coverage_pct"] = (
        100*s.n_evaluated_cells / s.n_planned_cells
    )

    return s


def check_summary(summary, old, out):
    new = summary[
        summary.model.eq("condition_iso")
        & summary.comparison.eq("own_support")
    ]

    keys = ["analysis", "validation", "band"]

    columns = [
        "MAE_mV",
        "mean_cell_RMSE_mV",
        "bias_mV",
        "worst_abs_mV",
        "n_planned_cells",
        "n_evaluated_cells",
        "n_grid",
        "n_target",
        "n_paired",
        "n_used",
    ]

    q = new[keys + columns].merge(
        old[keys + columns],
        on=keys,
        how="outer",
        suffixes=("_new", "_ref"),
        validate="one_to_one",
        indicator=True,
    )

    require(
        q["_merge"].eq("both").all(),
        "Referans özet kayıtları farklı.",
    )

    for c in columns:
        x = q[c + "_new"]
        y = q[c + "_ref"]

        q[c + "_abs_difference"] = abs(x-y)

        require(
            np.allclose(
                x, y,
                atol=1e-6,
                rtol=0,
                equal_nan=True,
            ),
            "Referans özet uyuşmuyor: " + c,
        )

    csv_save(q, out, "reference_summary_check.csv")

    print(
        "Referans MAE, RMSE, bias, en kötü hata "
        "ve kapsam sayımları doğrulandı."
    )


# ============================================================
# EŞLEŞTİRİLMİŞ HÜCRE BOOTSTRAP'I
# ============================================================

def bootstrap_pairs(pairs, out, reps=BOOTSTRAP_REPS):
    """
    Sabit out-of-fold hata farklarında koşul-tabakalı
    fiziksel hücre bootstrap'ı.

    İki yöntem için aynı hücreler birlikte örneklenir.
    Modeller yeniden fit edilmez.
    CI'lar noktasaldır ve mevcut test tahminlerine koşulludur.
    """
    require(
        reps >= 2000,
        "Nihai rapor için en az 2000 bootstrap kullanın.",
    )

    rng = np.random.default_rng(SEED)
    results = []
    draw_export = {}

    keys = [
        "analysis",
        "validation",
        "comparison",
        "left_model",
        "right_model",
    ]

    for idx, (key, g) in enumerate(
        pairs.groupby(keys, sort=True)
    ):
        e = (
            g[g.eligible]
            .sort_values(["Condition_ID", "sample_int"])
            .reset_index(drop=True)
        )

        row = dict(zip(keys, key))

        row.update(
            n_planned_cells=len(g),
            n_evaluated_cells=len(e),
            n_skipped_cells=len(g)-len(e),
            n_grid=int(g.n_grid.sum()),
            n_target=int(g.n_target.sum()),
            n_used=int(g.n_used.sum()),
            n_conditions=int(e.Condition_ID.nunique()),
        )

        row["target_coverage_pct"] = (
            100*row["n_used"]/row["n_target"]
            if row["n_target"] else np.nan
        )

        if not len(e):
            row.update(
                status="no_evaluable_cells",
                gain_right_mV=np.nan,
                CI95_low_mV=np.nan,
                CI95_high_mV=np.nan,
                bootstrap_reps=0,
            )

        else:
            left = e.left_MAE_mV.to_numpy()
            right = e.right_MAE_mV.to_numpy()
            delta = left-right

            require(
                np.isfinite(delta).all(),
                "Bootstrap farkları sonlu değil.",
            )

            require(
                not e.sample_int.duplicated().any(),
                "Aynı deneyde hücre birden çok kez sayılmış.",
            )

            L = np.zeros(reps)
            R = np.zeros(reps)

            strata = [
                part.index.to_numpy()
                for _, part in e.groupby(
                    "Condition_ID",
                    sort=True,
                )
            ]

            for ids in strata:
                take = rng.choice(
                    ids,
                    size=(reps, len(ids)),
                    replace=True,
                )

                L += left[take].sum(axis=1)
                R += right[take].sum(axis=1)

            L /= len(e)
            R /= len(e)

            draws = L-R
            draws[abs(draws) <= NUMERIC_TIE_MV] = 0.0

            low, high = np.quantile(
                draws,
                [0.025, 0.975],
            )

            gain = float(delta.mean())
            denominator = float(left.mean())

            if abs(gain) <= NUMERIC_TIE_MV:
                gain = 0.0

            row.update(
                status="ok_conditional_fixed_predictions",
                left_MAE_mV=denominator,
                right_MAE_mV=float(right.mean()),
                gain_right_mV=gain,
                gain_right_percent=(
                    100*gain/denominator
                    if denominator else np.nan
                ),
                CI95_low_mV=float(low),
                CI95_high_mV=float(high),
                CI_excludes_zero=bool(
                    low > NUMERIC_TIE_MV
                    or high < -NUMERIC_TIE_MV
                ),
                bootstrap_reps=reps,
                bootstrap_SD_mV=float(
                    np.std(draws, ddof=1)
                ),
                degenerate_bootstrap=bool(
                    np.ptp(draws) <= NUMERIC_TIE_MV
                ),
                n_cells_right_better=int(
                    (delta > NUMERIC_TIE_MV).sum()
                ),
                n_cells_tied=int(
                    (abs(delta) <= NUMERIC_TIE_MV).sum()
                ),
                n_cells_left_better=int(
                    (delta < -NUMERIC_TIE_MV).sum()
                ),
                n_singleton_strata=sum(
                    len(ids) == 1 for ids in strata
                ),
                max_abs_cell_gain_mV=float(
                    abs(delta).max()
                ),
                bootstrap_fraction_right_better=float(
                    (draws > NUMERIC_TIE_MV).mean()
                ),
                bootstrap_fraction_tied=float(
                    (abs(draws) <= NUMERIC_TIE_MV).mean()
                ),
            )

            label = "|".join(map(str, key))
            draw_export[label] = draws

        results.append(row)

        if (idx+1) % 20 == 0:
            print(
                f"Eşleştirilmiş bootstrap: "
                f"{idx+1} karşılaştırma tamamlandı.",
                flush=True,
            )

    np.savez_compressed(
        Path(out) / "bootstrap_gain_draws_mV.npz",
        **draw_export,
    )

    result = pd.DataFrame(results)

    csv_save(
        result,
        out,
        "paired_bootstrap_summary.csv",
    )

    return result


# ============================================================
# NİHAİ TEMSİLLER VE LUT REFERANS KONTROLÜ
# ============================================================

def export_final(datasets, old_lut, old_support, out):
    rows = []

    for analysis, d in datasets.items():
        frame = d.pivot(
            index="sample_int",
            columns="SoC",
            values="Voltage",
        ).reindex(columns=GRID)

        metadata = (
            d.drop_duplicates("sample_int")
            .set_index("sample_int")[
                [
                    "Condition_ID",
                    "Temperature",
                    "Rate",
                    "Charge_Rate",
                ]
            ]
        )

        frame = frame.join(metadata)

        audit = []
        fold = Fold(
            frame,
            analysis,
            "all_available_cells",
            audit,
        )

        for cid in range(1, 25):
            for model in MODELS:
                p, a, b, w = fold.predict(
                    model,
                    cid,
                    "final",
                )

                for soc in GRID:
                    rows.append({
                        "analysis": analysis,
                        "Condition_ID": cid,
                        "model": model,
                        "SoC": int(soc),
                        "prediction_V": float(p[soc]),
                        "n_training_cells_at_node": int(
                            a["support"][soc]
                        ),
                        "n_train_cells": a["n_train_cells"],
                        "fit_status": a["status"],
                    })

    lut = pd.DataFrame(rows)

    csv_save(
        lut,
        out,
        "all_representations_0_100.csv",
    )

    csv_save(
        lut[lut.SoC <= 90],
        out,
        "all_representations_0_90.csv",
    )

    final = lut[
        lut.analysis.eq("full_history")
        & lut.model.eq("condition_iso")
        & lut.SoC.le(90)
    ]

    arr = final.pivot(
        index="Condition_ID",
        columns="SoC",
        values="prediction_V",
    ).to_numpy()

    support = final.pivot(
        index="Condition_ID",
        columns="SoC",
        values="n_training_cells_at_node",
    ).to_numpy()

    require(
        arr.shape == old_lut.shape
        and np.array_equal(
            np.isfinite(arr),
            np.isfinite(old_lut),
        ),
        "Nihai LUT şekli/sonlu maskesi değişti.",
    )

    require(
        np.allclose(
            arr,
            old_lut,
            rtol=0,
            atol=1e-9,
            equal_nan=True,
        ),
        "Nihai LUT değerleri değişti.",
    )

    require(
        np.array_equal(support, old_support),
        "Nihai LUT hücre desteği değişti.",
    )

    np.save(
        Path(out) / "final_condition_LUT_float64.npy",
        arr,
    )

    np.save(
        Path(out) / "final_condition_LUT_float32.npy",
        arr.astype("float32"),
    )

    qa = {
        "final_lut_reference_max_difference_mV": float(
            np.nanmax(abs(arr-old_lut))*1000
        ),
        "final_lut_shape": list(arr.shape),
        "final_lut_finite_nodes": int(
            np.isfinite(arr).sum()
        ),
    }

    # İzotonik düzeltmenin düğüm bazlı büyüklüğü ve desteği.
    pair = lut[
        lut.model.isin(["condition_raw", "condition_iso"])
    ].pivot(
        index=["analysis", "Condition_ID", "SoC"],
        columns="model",
        values="prediction_V",
    ).reset_index()

    supports = lut[lut.model.eq("condition_iso")][
        [
            "analysis",
            "Condition_ID",
            "SoC",
            "n_training_cells_at_node",
        ]
    ]

    pair = pair.merge(
        supports,
        on=["analysis", "Condition_ID", "SoC"],
        validate="one_to_one",
    )

    pair["isotonic_minus_raw_mV"] = (
        1000*(pair.condition_iso-pair.condition_raw)
    )

    csv_save(
        pair,
        out,
        "isotonic_node_adjustments.csv",
    )

    (Path(out) / "LUT_reference_check.json").write_text(
        json.dumps(qa, indent=2),
        encoding="utf-8",
    )

    return qa


# ============================================================
# ÇIKTI AÇIKLAMASI
# ============================================================

README = """
ADIM 5: KOŞUL BİLGİSİ VE İZOTONİK DÜZELTME KARŞILAŞTIRMASI

MODELLER
global_iso: Tek ortak izotonik eğri.
temperature_iso: Yalnız sıcaklığa bağlı izotonik eğriler.
rate_iso: Yalnız deşarj hızına bağlı izotonik eğriler.
condition_raw: Koşula özgü hücre medyanı; izotonik uygulanmaz.
condition_iso: Koşula özgü destek-ağırlıklı izotonik LUT.

Her eğitim satırı bir fiziksel hücrenin medyan profilidir.
Azaltılmış modeller uygun eğitim hücrelerini doğrudan havuzlar.
Koşullara eşit ağırlık uygulanmaz; her fiziksel hücre bir satırdır.
İlk üç model ve condition_iso aynı izotonik kurallarını kullanır.
condition_raw aynı eğitim hücrelerinin düğüm medyanını kullanır.
Ham/izotonik modellerinde minimum 2 destekli fit düğümü şartı aynıdır.

FİT VE DOĞRULAMA
Fit: 0..100. Değerlendirme: 0..90 dahil.
Düğümde en az bir eğitim hücresi gerekir; sıfır destek NaN kalır.
LOCO bütün eğitim havuzundan test hücresini çıkarır.
Faktör tutma bütün tutulan düzeyi bütün modellerin eğitiminden çıkarır.
25 ve 45 derece sıcaklık tutmaları ayrı deneylerdir.
Tutulan faktörü kullanan modeller iki komşu eğriyi doğrusal birleştirir.
Tutulan faktörü kullanmayan modeller kalan eğitimde kendi eğrisini kurar.
Komşulardan biri destek vermiyorsa taşıma veya ekstrapolasyon yapılmaz.

ANA SONUÇ
comparison=common_five, band=all_0_90_inclusive.
Beş model aynı hücre–SoC düğümlerinde puanlanır.
Her hücre/bant için en az 5 sonlu hedef–tahmin çifti gerekir.
own_support her yöntemin kendi kapsamını gösterir.
pairwise_common iki yöntemin ortak kapsamını gösterir.
Farklı maskelerdeki MAE'ler doğrudan iyileşme kanıtı sayılmaz.
MAE, değerlendirilebilir hücre MAE'lerinin aritmetik ortalamasıdır.
mean_cell_RMSE, hücre RMSE'lerinin ortalamasıdır; pooled RMSE değildir.
Hedef hücre medyan gerilim profilidir.
Bireysel çevrim veya çevrimiçi SoC kestirim hatası değildir.
# %80 bant kapsam filtresi LUT girdilerine ayrıca uygulanmamıştır.
Yardımcı hata bantları: [0,20), [20,60), [60,90].
Son hata bandı 90'ı içerir; kontrast analizindeki [60,90) farklıdır.

KAZANÇ YÖNÜ
gain_right_mV = MAE(left_model) - MAE(right_model).
Pozitif değer sağdaki yöntemin daha küçük MAE verdiğini gösterir.
condition_raw -> condition_iso karşılaştırması izotonik katkıdır.
condition_raw kaynak ham gerilim kayıtlarını ifade etmez.

BOOTSTRAP
2000 koşul-tabakalı, eşleştirilmiş fiziksel hücre örneklemesi.
Modeller bootstrap içinde yeniden fit edilmez.
Aralıklar mevcut out-of-fold tahminlere koşulludur.
Paylaşılan eğitim kümelerinden kaynaklanan bağımlılığı veya tüm eğitim
belirsizliğini kapsadıkları iddia edilmez.
Tek hücreli tabaka ek değişkenlik üretmez.
degenerate_bootstrap=True, bootstrap farklarının değişkenliğinin
yok/çok az olduğunu gösterir; sıfır genişlikli CI genel kesinlik kanıtı değildir.
Aralıklar noktasaldır; çoklu karşılaştırma kontrolü uygulanmaz.
Bootstrap işaret oranı p-değeri değildir.
Faktör tutmada koşullar sabit tutulur.
Aralıklar yeni faktör düzeylerine evrensel aktarımı veya iki sıcaklık
katının popülasyon belirsizliğini tahmin etmez.
Değerlendirilemeyen hücrelere sıfır hata atanmaz.

REFERANS VE KAPSAM
Önceki unified düğüm tahminleri, hedefleri, bütün bant özetleri ve nihai
LUT tolerans içinde yeniden üretilmeden karşılaştırma tamamlanmaz.
Girdi SHA256 kontrolü zorunludur.
Upstream çevrim yeniden kurma ve akım hedefi seçimi yeniden çalıştırılmaz.
Yalnız profil düzeyindeki eğitim/test ayrımı denetlenir.
Tüm upstream işlemlerin training-only olduğu iddia edilmez.
Nihai LUT tam geçmişten üretilir; 108'e yaş standardizasyonu yapılmaz.
SoC, çevrim sonu kapasitesine göre normalize edilmiş çevrimdışı koordinattır.

ANA DOSYALAR
MAIN_common_five.csv
MAIN_gains_vs_condition_iso.csv
summary.csv
summary_by_condition.csv
cell_metrics.csv
paired_cell_MAE_differences.csv
paired_bootstrap_summary.csv
bootstrap_gain_draws_mV.npz
node_predictions.csv.gz
node_exclusion_reasons.csv
training_fit_audit.csv
reference_node_check.csv
reference_summary_check.csv
LUT_reference_check.json
isotonic_node_adjustments.csv
"""


# ============================================================
# ANALİZİ ÇALIŞTIR
# ============================================================

def run(uploaded, out):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)

    require(
        int(BOOTSTRAP_REPS) == BOOTSTRAP_REPS
        and BOOTSTRAP_REPS >= 2000,
        "Bootstrap tekrar sayısı en az 2000 olmalı.",
    )

    (
        datasets,
        old_nodes,
        old_summary,
        old_lut,
        old_support,
        hashes,
        provenance,
        reference_hash,
    ) = load_inputs(uploaded)

    config = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "models": list(MODELS),
        "bootstrap_reps": BOOTSTRAP_REPS,
        "seed": SEED,
        "min_eval_nodes": MIN_EVAL_POINTS,
        "input_sha256": hashes,
        "input_source": provenance,
        "reference_zip_sha256": reference_hash,
        "pool_weight": "one physical-cell median profile per row",
        "main_mask": "common_five",
        "fit_soc": [0, 100],
        "evaluate_soc": [0, 90],
        "bootstrap": (
            "condition-stratified paired cells; "
            "fixed out-of-fold errors; pointwise CI"
        ),
    }

    (out / "run_config.json").write_text(
        json.dumps(config, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    (out / "README.txt").write_text(
        README,
        encoding="utf-8",
    )

    csv_save(META, out, "condition_map.csv")

    inventory = []
    all_nodes = []
    all_audits = []

    for name, d in datasets.items():
        print(
            f"\n{name}: {d.sample_int.nunique()} hücre, "
            f"{d.Condition_ID.nunique()} koşul.",
            flush=True,
        )

        for cid in range(1, 25):
            g = d[d.Condition_ID.eq(cid)]

            inventory.append({
                "analysis": name,
                "Condition_ID": cid,
                "n_cells": g.sample_int.nunique(),
                "n_finite_nodes_0_90": int(
                    np.isfinite(
                        g.loc[g.SoC <= 90, "Voltage"]
                    ).sum()
                ),
            })

        nodes, audit = predict_dataset(d, name)
        all_nodes.append(nodes)
        all_audits.append(audit)

    nodes = pd.concat(all_nodes, ignore_index=True)

    csv_save(
        pd.DataFrame(inventory),
        out,
        "input_inventory.csv",
    )

    check_reference(nodes, old_nodes, out)

    print(
        "Hücre hata ve kapsam tabloları hesaplanıyor...",
        flush=True,
    )

    metrics, pairs = compute_metrics(nodes)
    summary = aggregate(metrics)

    check_summary(summary, old_summary, out)

    qa = export_final(
        datasets,
        old_lut,
        old_support,
        out,
    )

    csv_save(
        nodes,
        out,
        "node_predictions.csv.gz",
    )

    csv_save(
        pd.concat(all_audits, ignore_index=True),
        out,
        "training_fit_audit.csv",
    )

    csv_save(metrics, out, "cell_metrics.csv")
    csv_save(summary, out, "summary.csv")

    csv_save(
        aggregate(metrics, condition=True),
        out,
        "summary_by_condition.csv",
    )

    csv_save(
        pairs,
        out,
        "paired_cell_MAE_differences.csv",
    )

    exclusions = []

    for model in MODELS:
        reason_column = model + "_reason"

        g = (
            nodes.groupby([
                "analysis",
                "validation",
                "Condition_ID",
                reason_column,
            ])
            .size()
            .rename("n_nodes")
            .reset_index()
            .rename(columns={reason_column: "reason"})
        )

        g["model"] = model
        exclusions.append(g)

    csv_save(
        pd.concat(exclusions, ignore_index=True),
        out,
        "node_exclusion_reasons.csv",
    )

    bootstrap = bootstrap_pairs(
        pairs,
        out,
        reps=BOOTSTRAP_REPS,
    )

    main_summary = summary[
        summary.comparison.eq("common_five")
        & summary.band.eq("all_0_90_inclusive")
    ]

    csv_save(
        main_summary,
        out,
        "MAIN_common_five.csv",
    )

    main_gains = bootstrap[
        bootstrap.comparison.eq("common_five")
        & bootstrap.left_model.isin([
            "global_iso",
            "temperature_iso",
            "rate_iso",
            "condition_raw",
        ])
        & bootstrap.right_model.eq("condition_iso")
    ]

    csv_save(
        main_gains,
        out,
        "MAIN_gains_vs_condition_iso.csv",
    )

    print("\nANA KARŞILAŞTIRMA / AYNI HÜCRE VE DÜĞÜMLER:")

    print(
        main_summary[[
            "analysis",
            "validation",
            "model",
            "n_evaluated_cells",
            "n_used",
            "target_coverage_pct",
            "MAE_mV",
            "mean_cell_RMSE_mV",
        ]].to_string(index=False)
    )

    print(
        "\nCONDITION_ISO KAZANCI:"
        "\nPozitif değer = condition_iso daha düşük MAE."
    )

    print(
        main_gains[[
            "analysis",
            "validation",
            "left_model",
            "gain_right_mV",
            "CI95_low_mV",
            "CI95_high_mV",
            "n_evaluated_cells",
            "target_coverage_pct",
            "degenerate_bootstrap",
        ]].to_string(index=False)
    )

    print("\nNihai LUT referans kontrolü:", qa)

    return main_summary, main_gains


# ============================================================
# MANUEL YÜKLEME VE OTOMATİK ZIP İNDİRME
# ============================================================

def main():
    from google.colab import files

    print(
        "MANUEL YÜKLEME:\n"
        "1. Önceki SoC_unified_....zip dosyasını seçin (zorunlu).\n"
        "2. İsterseniz aynı anda iki CSV'yi de seçebilirsiniz:\n"
        "   cell_profiles_nominal_concordant.csv\n"
        "   broad_52_132_cell_profiles.csv\n\n"
        "CSV seçmezseniz ZIP içindeki girdiler kullanılacak.\n"
        "Bant/yaş duyarlılığı ZIP dosyalarını seçmeyin."
    )

    uploaded = files.upload()

    require(
        bool(uploaded),
        "Dosya seçilmedi.",
    )

    out = Path(
        tempfile.mkdtemp(prefix="SoC_step5_ablation_")
    )

    try:
        run(uploaded, out)

    except Exception as exc:
        (out / "ERROR.txt").write_text(
            type(exc).__name__ + ": " + str(exc),
            encoding="utf-8",
        )

        print(
            "\nÇalışma durduruldu."
            "\nKısmi sonuçlar nihai değildir."
            "\nTanı klasörü:",
            out,
        )

        raise

    archive = shutil.make_archive(
        str(out),
        "zip",
        root_dir=out,
    )

    print("\nTAMAMLANDI. Çıktı:", archive)
    print(
        "Önce bakılacak dosyalar:\n"
        "  MAIN_common_five.csv\n"
        "  MAIN_gains_vs_condition_iso.csv\n"
        "  README.txt"
    )

    files.download(archive)


# Incomplete exported guard omitted.

# ---------- LOCAL, NONDESTRUCTIVE STAGE RUNNER ----------
path=require_path('/content/SoC_unified_ConditionvCodes2.zip')
out=stage_dir('05_five_representations')
run({path.name:path.read_bytes()},out)
import pandas as pd
for name in ['MAIN_common_five.csv','MAIN_gains_vs_condition_iso.csv']:
    d=pd.read_csv(out/name)
    print(name,len(d),'rows')
stage_zip('05_five_representations','SoC_step5_ablation_')
